# AI-Powered Grievance Classification System — Bengaluru
### Automatically route citizen complaints to the correct civic authority and predict their urgency

---

**Project Overview**

Bengaluru receives thousands of citizen grievances daily across multiple civic agencies — BBMP, BWSSB, BESCOM, BTP, and others. Manually triaging these complaints is slow, error-prone, and costly.

This notebook builds an end-to-end NLP pipeline that:
1. **Routes complaints** to the correct civic authority (civic agency classification)
2. **Prioritises complaints** by predicted urgency: Low → Medium → High → Critical (severity classification)

Both tasks use the same complaint text as input. We benchmark classical ML models (Logistic Regression, LinearSVC, Random Forest, Multinomial Naive Bayes) against deep learning approaches (DistilBERT fine-tuning, BiLSTM).

---

**Notebook Structure**

| Section | Description |
|---------|-------------|
| 1 | Data Retrieval & Cleaning |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Civic Agency Classification — Preprocessing, Augmentation & Training |
| 4 | Civic Agency Classification — Final Dataset & Model Saving |
| 5 | Severity Classification — Preprocessing, Augmentation, Training & Inference |

---


## 1  Data Retrieval & Cleaning

### 1.1  Imports & Environment Setup

All standard libraries, ML frameworks, and NLP tools are loaded here.  
GPU memory is capped at 75 % to prevent OOM errors during augmentation and fine-tuning.


In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import re, copy, json, glob, gzip, random, shutil, logging, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path

# ── Data Manipulation & Visualisation ────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import json
import requests
import time

# ── Environment & Database ───────────────────────────────────────────────────
from dotenv import load_dotenv
from sqlalchemy import create_engine

# ── NLP Tools ────────────────────────────────────────────────────────────────
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.augmenter.sentence as nas

# Prevent accidental downloads inside a controlled environment
nltk.download = lambda *args, **kwargs: True

# ── Scikit-Learn ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import StratifiedKFold, ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, precision_recall_fscore_support,
    classification_report, precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_class_weight

# ── PyTorch & Hugging Face ────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.75, device=0)
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# ── Scipy ─────────────────────────────────────────────────────────────────────
from scipy.special import softmax
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu
from scipy.stats import skew

# -- Open AI API --------------------------------------------------------------
from openai import OpenAI
import google.generativeai as genai
import vertexai
from vertexai.generative_models import GenerativeModel

# ── Project Root ─────────────────────────────────────────────────────────────
# Assumes this notebook lives one level inside the project (e.g. /notebook/)
PROJECT_ROOT = Path.cwd().parent
CHARTS_DIR   = PROJECT_ROOT / "charts_and_graphs"
CHARTS_DIR.mkdir(exist_ok=True)
print(f"Project root : {PROJECT_ROOT}")
print(f"Charts folder: {CHARTS_DIR}")


### 1.2  Database Connection

Credentials are stored in a `.env` file at `<project_root>/src/.env` — **never hard-coded**.


In [ ]:
load_dotenv(dotenv_path=PROJECT_ROOT / "src" / ".env", override=True)

DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('user')}:{os.getenv('password')}"
    f"@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as conn:
        print("✅ Database connection successful.")
except Exception as e:
    print(f"❌ Connection failed: {e}")

### 1.3  Load Full Dataset


In [ ]:
df = pd.read_sql("SELECT * FROM bbmc_final_data;", engine)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.columns

In [ ]:
df= df[['created_at','description','civic_agency_id', 'civic_agency_title', 
        'complaints_length', 'severity_score', 
        'confidence_score', 'severity_reason']]

### 1.4  Missing-Value Analysis

We check the percentage of nulls across all columns before deciding which rows to drop.


In [ ]:
df_null_pct = df.isna().mean() * 100
print("Null percentage per column (non-zero only):")
print(df_null_pct[df_null_pct > 0].sort_values(ascending=False))

In [ ]:
df= df.drop(columns=['civic_agency_id'])
df.columns

### 1.5  Drop Null Records

Both `description` (the complaint text) and `severity` (the target label) are mandatory.
Any row missing either field is removed.


In [ ]:
print(f"Rows before cleaning : {df.shape[0]:,}")
df.dropna(subset=["description", "civic_agency_title", "severity_score"], inplace=True)
print(f"Rows after dropping nulls : {df.shape[0]:,}")

### 1.6  Standardise Civic Agency Names

Some agencies appear under both their full name and acronym (e.g. *BBMP* and *Bruhat Bengaluru Mahanagara Palike*), causing artificially split complaint counts.  
We consolidate all variants into a single canonical acronym, and merge very sparse agencies into the closest parent to reduce label sparsity.


In [ ]:
df['civic_agency_title'].value_counts()

In [ ]:
# ── Step 1: full name → acronym ───────────────────────────────────────────────
agency_alias_map = {
    "Bruhat Bengaluru Mahanagara Palike":       "BBMP",
    "Bangalore Traffic Police":                 "BTP",
    "Bangalore Water Supply And Sewerage Board": "BWSSB",
    "Karnataka State Pollution Control Board":  "KSPCB",
    "Bangalore Electricity Supply Company":     "BESCOM",
}
df["civic_agency_title"] = df["civic_agency_title"].replace(agency_alias_map)

# ── Step 2: merge sparse agencies into parent categories ─────────────────────
agency_consolidation_map = {
    "BDA":   "BBMP",       # urban infrastructure overlap
    "BMTC":  "Transport",  # public transport
    "KSRTC": "Transport",
}
df["civic_agency_title"] = df["civic_agency_title"].replace(agency_consolidation_map)

print(f"Unique civic agencies after consolidation: {df['civic_agency_title'].nunique()}")
print(df["civic_agency_title"].value_counts())

### 1.7  Final Deduplication & Index Reset


In [ ]:
df = (
    df.dropna(subset=["description", "severity_score"])
      .drop_duplicates()
      .reset_index(drop=True)
)
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(2)

---
## 2  Exploratory Data Analysis (EDA)

### 2.1  Complaint Length Distribution

We measure the word count of each complaint and examine how it varies across severity categories.  
Very short complaints (fewer than ~14 words) may lack enough context to reliably predict severity — this informs our augmentation threshold.


In [ ]:
PATH_PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

df.to_csv(PATH_PROCESSED_DATA / "bbmc_data_v2.csv", index=False)

df.head(2)

Bucketing the Severity:

In [ ]:
def get_severity(score):
    if score >= 90:
        return "Critical"
    elif score >= 80:
        return "High"
    elif score >= 50:
        return "Medium"
    elif score >= 1:
        return "Low"
    else:
        return "Non-Grievance"

df["severity"] = df["severity_score"].apply(get_severity)

In [ ]:
# Sort chronologically for time-based analysis later
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df = df.sort_values("created_at", ascending=True).reset_index(drop=True)

# Word-count feature
df["complaint_length"] = df["description"].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
sns.histplot(df["complaint_length"], bins=100, kde=True, ax=axes[0])
axes[0].axvline(df["complaint_length"].mean(),   linestyle="--",
                label=f'Mean ({df["complaint_length"].mean():.1f})')
axes[0].axvline(df["complaint_length"].median(), linestyle="-",
                label=f'Median ({df["complaint_length"].median():.0f})')
axes[0].set_title("Overall Complaint Word-Length Distribution")
axes[0].set_xlabel("Word Count"); axes[0].set_ylabel("Frequency")
axes[0].legend()

# By severity
sns.boxplot(x="severity", y="complaint_length", data=df,
            order=["Low", "Medium", "High", "Critical"], ax=axes[1])
axes[1].set_title("Complaint Length by Severity Category")
axes[1].set_xlabel("severity"); axes[1].set_ylabel("Word Count")

plt.tight_layout()
plt.savefig(CHARTS_DIR / "2.1_complaint_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Descriptive statistics per severity
print("Complaint length statistics by severity:\n")
print(df.groupby("severity")["complaint_length"].describe().round(2))

# KDE overlay
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x="complaint_length", hue="severity", fill=True)
plt.title("Complaint Length Density by Severity")
plt.xlabel("Word Count")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "2.1b_complaint_length_kde.png", dpi=150, bbox_inches="tight")
plt.show()

Detecting the outlier based on the complaint length (unusually short or long complaints):

In [ ]:
def detect_outlier(feature):
    Q1 = feature.quantile(0.25)
    Q3 = feature.quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return lower_bound, upper_bound

In [ ]:
lower_bound, upper_bound = detect_outlier(df["complaints_length"])

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

As the dataset is highly skewed, this si why the lowerbound is apperaing to be negative, lets run a test for detecting the skewness of the tranformation figures and then decide a tranformation based on that:

In [ ]:
print("Original:", skew(df["complaints_length"]))
print("Log:", skew(np.log1p(df["complaints_length"])))
print("Sqrt:", skew(np.sqrt(df["complaints_length"])))
print("Cube Root:", skew(np.cbrt(df["complaints_length"])))

In [ ]:
log_length = np.log1p(df["complaints_length"])

Q1 = log_length.quantile(0.25)
Q3 = log_length.quantile(0.75)

IQR = Q3 - Q1

lower_log = Q1 - 1.5 * IQR
upper_log = Q3 + 1.5 * IQR

print("Log Lower Bound:", lower_log)
print("Log Upper Bound:", upper_log)

In [ ]:
lower_original = np.expm1(lower_log)
upper_original = np.expm1(upper_log)

print("Original Lower Bound:", lower_original)
print("Original Upper Bound:", upper_original)

So as per the log transformed ersults, the lowerbound is 1, so that means we can discard the complaints having length of 0 words that contains only special charcters or symbols, as tehy carry no menaingful conext.

In [ ]:
log_length = np.log1p(df["complaints_length"])

stats.probplot(log_length, dist="norm", plot=plt)

plt.title("Q-Q Plot of log1p(complaints_length)")
plt.show()

A comparison of orginal vs log tranformed distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(df["complaints_length"], bins=50, color="blue", edgecolor="white")
ax[0].set_title("Original Distribution")

ax[1].hist(log_length, bins=50, color="blue", edgecolor="white")
ax[1].set_title("After Log1p Transformation")

fig.suptitle(
    "Effect of Log Transformation on Complaint Length",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

Based on the data and the explainibility we will keep the original numbers as criteria for detecting the too long or too short complaints. So as per that the lower bound is 0 but we will keep that as 1 as 0 complaint length has no useful context and the upper bound is 108 for flagging a complaint length as an outlier.

In [ ]:
# Inspect the shortest complaints — these are candidates for augmentation
df_short = df[df["complaint_length"] < 108][["description", "complaint_length", "severity"]].head(10)
print("Sample complaints under 110 words:")
display(df_short)

In [ ]:
# Percentile breakdown — useful for choosing the 256-word summarisation threshold
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_stats = (
    df.groupby("severity")["complaint_length"]
      .quantile(percentiles)
      .unstack()
      .round(2)
)
pct_stats.columns = [f"{int(p*100)}%" for p in percentiles]
print("Complaint length percentiles by severity:\n")
print(pct_stats)

In [ ]:
df["complaints_length"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999]
)

So based on our analysis 95% complaints are below complaint_length of 108.

### 2.2  Grievance Volume Over Time

We look at how total complaints — broken down by severity — have changed year over year.


In [ ]:
df["year"] = df["created_at"].dt.year

palette = {
    "Non-Grievance": "grey",
    "Low": "green",
    "Medium": "orange",
    "High": "red",
    "Critical": "blue"
}

plt.figure(figsize=(10, 6))
sns.countplot(
    data=df,
    x="year",
    hue="severity",
    palette=palette
)
plt.title("Number of Grievances by Severity per Year")
plt.xlabel("Year"); plt.ylabel("Complaint Count")
plt.legend(title="Severity")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "2.2_grievances_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3  Complaint Distribution per Civic Agency

Examining complaint volume across agencies, overall and by year, helps identify disproportionately burdened agencies and informs routing model complexity.


In [ ]:
print(f"Total civic agencies (after consolidation): {df['civic_agency_title'].nunique()}")
print("\nComplaint count per agency:")
print(df["civic_agency_title"].value_counts())

In [ ]:
palette = {
    "Non-Grievance": "grey",
    "Low": "green",
    "Medium": "orange",
    "High": "red",
    "Critical": "blue"
}
# Year-by-year severity breakdown per agency (percentage)
for agency in df["civic_agency_title"].dropna().unique():
    data  = df[df["civic_agency_title"] == agency]
    pivot = data.pivot_table(index="year", columns="severity", aggfunc="size", fill_value=0)
    if pivot.empty:
        continue
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    pivot_pct.plot(kind="bar", stacked=False, figsize=(8, 4), color=[palette[col] for col in pivot_pct.columns])
    plt.ylim(0, 100)
    plt.title(f"{agency} — Complaint Severity Distribution (%) by Year")
    plt.xlabel("Year"); plt.ylabel("% of Complaints")
    plt.legend(title="Severity")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / f"2.3_{agency}_severity_by_year.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 3  Civic Agency Classification — Preprocessing, Augmentation & Training

**Objective:** Given the free-text description of a complaint, predict which civic agency (*BBMP, BWSSB, BESCOM, BTP, Transport, …*) should handle it.

**Training strategy:** 5-fold stratified cross-validation. Each training fold is augmented independently before fitting; the model is evaluated on the untouched validation fold to prevent data leakage.


In [ ]:
df= df[['description','civic_agency_title','severity','severity_score','complaint_length']]
df.head()

### 3.1  Text Preprocessing

For classical ML models each complaint passes through:
1. Lowercasing
2. Boilerplate removal (greetings, sign-offs)
3. URL removal
4. Non-alphabetic character removal
5. Whitespace normalisation
6. Stopword removal + Lemmatisation


In [ ]:
# ── NLTK setup ────────────────────────────────────────────────────────────────
local_nltk_path = PROJECT_ROOT / "data" / "nltk"
local_nltk_path.mkdir(parents=True, exist_ok=True)
nltk.data.path.insert(0, str(local_nltk_path))

try:
    STOPWORDS   = set(stopwords.words("english"))
    _lemmatizer = WordNetLemmatizer()
    print("✅ NLTK stopwords and lemmatizer loaded.")
except LookupError:
    raise RuntimeError(f"❌ NLTK stopwords not found. Check: {local_nltk_path}")

nltk.download = lambda *args, **kwargs: True
logging.getLogger("nltk").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

_BOILERPLATE = [
    r"dear sir.*?", r"dear madam.*?",
    r"regards.*?",  r"sent from my.*?", r"thank you.*?",
]

def preprocess_classical_ml(text: str) -> str:
    text = str(text).lower()
    for pat in _BOILERPLATE:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)
    text = re.sub(r"http[s]?://\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text  # No lemmatization

# Sanity check
sample = "Dear Sir, The road near my house has deep potholes. Regards, Citizen"
print(f"Before : {sample}")
print(f"After  : {preprocess_classical_ml(sample)}")

Though for NLP based BERT models:

For BERT-based models, each complaint passes through:

1. URL removal
2. Removal of complaint/ticket/reference IDs (if present)
3. Removal of obvious boilerplate metadata (e.g., "Sent from my iPhone")
4. Whitespace normalization
5. Preservation of original casing, punctuation, stopwords, and word forms to retain contextual    information for the BERT tokenizer

In [ ]:
_BOILERPLATE = [
    r"sent from my.*?",
]

def preprocess_nlp_bert(text: str) -> str:
    text = str(text)

    # Remove URLs
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)

    # Remove complaint/ticket/reference IDs
    text = re.sub(
        r"(complaint|ticket|reference)\s*(id|number|no)?\s*[:\-]?\s*\w+",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove boilerplate signatures
    for pat in _BOILERPLATE:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Sanity check
sample = "Dear Sir, The road near my house has deep potholes. Regards, Citizen"
print(f"Before : {sample}")
print(f"After  : {preprocess_nlp_bert(sample)}")

### 3.3  N-gram Frequency Analysis & Word Cloud

Inspecting the top unigrams, bigrams, and trigrams confirms the preprocessed text retains domain-relevant vocabulary (e.g. *road, garbage, water, drainage*) and reveals any remaining noise.


In [ ]:
# Apply preprocessing to the full corpus for EDA
df["clean"] = df["description"].apply(preprocess_classical_ml)

def get_ngram_freq(text_series, ngram_range=(2, 2), top_k=20, min_df=10):
    if text_series.empty:
        return pd.DataFrame(columns=["ngram", "frequency"])
    vec = CountVectorizer(ngram_range=ngram_range, min_df=min_df, stop_words="english")
    X_v = vec.fit_transform(text_series)
    counts = X_v.sum(axis=0).A1
    return (
        pd.DataFrame({"ngram": vec.get_feature_names_out(), "frequency": counts})
          .sort_values("frequency", ascending=False).head(top_k)
    )

def plot_ngram(df_ngram, title):
    if df_ngram.empty:
        print(f"No data for: {title}"); return
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    ax = sns.barplot(x="frequency", y="ngram", data=df_ngram, palette="viridis")
    plt.title(title, fontsize=14, loc="left")
    plt.xlabel("Frequency"); plt.ylabel("")
    for container in ax.containers:
        ax.bar_label(container, padding=3)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    # savefig must come BEFORE show() to avoid saving a blank figure
    plt.savefig(CHARTS_DIR / f"3.1_{title.replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

for ngram_range, title in [((1,1),"Top Unigrams"), ((2,2),"Top Bigrams"), ((3,3),"Top Trigrams")]:
    plot_ngram(get_ngram_freq(df["clean"], ngram_range=ngram_range), title)

In [ ]:
all_text = " ".join(df["clean"])
wc = WordCloud(width=1000, height=500, background_color="white",
               max_words=200, colormap="viridis").generate(all_text)
plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud — Preprocessed Complaint Corpus", fontsize=16)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "3.2_word_cloud.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4  Data Augmentation Setup

Class imbalance causes under-represented agencies to be poorly classified. Three complementary strategies are used:

- **Contextual word substitution** (DistilBERT) — replaces words with semantically similar alternatives
- **Synonym substitution** (WordNet) — lightweight and deterministic
- **Spelling augmentation** — injects minor typo-style noise for robustness

Complaints ≥ 256 words are **summarised** with T5 instead, as word-level substitution on very long texts adds little diversity.

**Augmentation intensity scales with class size:**

| Class size | Multiplier |
|-----------|-----------|
| > 2000 | skip |
| 1001–2000 | ×1 |
| 501–1000 | ×2 |
| 1-500 | ×3 |


### 3.5  Preprocessing & Augmentation — Civic Agency Folds


In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # AUGMENTATION SETUP 1
# # ─────────────────────────────────────────────────────────────────────────────
# #
# # Pipeline overview
# # -----------------
# # Called ONCE per fold on the fold's training slice (after train/test split,
# # before StratifiedKFold). Originals are always preserved; augmentation only
# # ADDS rows.
# #
# # Step 1 — Class-level tier (controls repeat count for ALL paths):
# #   count 1000-2000  ->  x1   
# #   count 500-1000   ->  x2
# #   count 1-500      ->  x3   
# #
# # Step 2 — Per-sample routing by complaint_length (raw word count):
# #   complaint_length > 15  ->  Classical augmenters (contextual word
# #                              embedding substitution once on the original complaint description,
# #                              this is a must step for all the complaints.)
# #                              Spelling Noise once on the original complaint description.
# #                              T5 abstractive summary (optional only if the complaint description
# #                              length is above 140 words on the roiginal sentence).
# #                              in one pass -> up to 1 new texts.
# #   complaint_length < 15  ->  Gemini sentence-level rewriting ->
# #                              up to 2 paraphrases per call.
# #
# # ROW-LEVEL CHECKPOINT
# # ---------------------
# # augment_dataset() accepts an optional `checkpoint_path` (Path | None).
# #   * On first run  : creates the JSONL file and appends one line per
# #     original row IMMEDIATELY after that row finishes processing.
# #   * On resume     : reads the file, recovers all previously generated
# #     synthetic texts, skips those row-indices, and continues from the
# #     first unprocessed row.
# #   * On crash      : at most ONE row of work is lost.  All earlier rows
# #     are already safely on disk.
# #   * After a fold completes successfully the caller deletes the checkpoint
# #     file so the directory stays clean.
# # ─────────────────────────────────────────────────────────────────────────────

# random.seed(43)
# np.random.seed(43)

# _device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Augmentation device: {_device}")

# # ── Gemini client ─────────────────────────────────────────────────────────────
# import vertexai
# from vertexai.generative_models import GenerativeModel

# vertexai.init(project="ai-grievance-sandi", location="us-central1")
# _gemini = GenerativeModel("gemini-2.5-flash")
# print("[OK] Gemini client initialised via Vertex AI (project: ai-grievance-sandi)")

# # ── Classical augmenters (long-text path) ─────────────────────────────────────
# _AUG_CONTEXTUAL = naw.ContextualWordEmbsAug(
#     model_path = "distilbert-base-uncased",
#     action     = "substitute",
#     device     = _device,
#     aug_p      = 0.10,
#     stopwords  = STOPWORDS - {"not", "no", "never", "against", "without"},
# )

# _AUG_SPELLING = naw.SpellingAug(aug_p=0.01)

# _AUG_SUMMARY = nas.AbstSummAug(
#     model_path = "t5-base",
#     device     = _device,
# )

# # Index 0 -> contextual word embedding substitution
# # Index 1 -> spelling noise
# # Index 2 -> T5 abstractive summary
# _CLASSICAL_AUGMENTERS: list = [_AUG_CONTEXTUAL, _AUG_SPELLING, _AUG_SUMMARY]


# # ─────────────────────────────────────────────────────────────────────────────
# # HELPERS
# # ─────────────────────────────────────────────────────────────────────────────

# def _safe_augment(augmenter, text: str) -> str | None:
#     # Run a single nlpaug augmenter; return a clean string or None.
#     try:
#         result = augmenter.augment(text)
#         result = result[0] if isinstance(result, list) else result
#         result = str(result).strip() if result else None
#         return result if result and result.lower() != "nan" else None
#     except Exception:
#         return None


# def _deduplicate(texts: list[str], original: str) -> list[str]:
#     # Remove duplicates and exact matches to the original (case-insensitive).
#     seen   = {original.lower()}
#     unique = []
#     for t in texts:
#         key = t.lower()
#         if key not in seen:
#             seen.add(key)
#             unique.append(t)
#     return unique


# def _clean_df(X: list, y: list, severity: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
#     frame = pd.DataFrame({"x": X, "y": y, "severity": severity}).dropna()
#     frame["x"] = frame["x"].astype(str).str.strip()
#     frame = frame[(frame["x"] != "") & (frame["x"].str.lower() != "nan")]
#     frame = frame.drop_duplicates()
#     return frame["x"].values, frame["y"].values, frame["severity"].values


# # ─────────────────────────────────────────────────────────────────────────────
# # PATH A — CLASSICAL AUGMENTATION (complaint_length >= 15)
# # ─────────────────────────────────────────────────────────────────────────────

# def classical_augment(text: str) -> list[str]:
#     # Apply all three classical augmenters independently to one text.
#     # Returns up to 1 new strings; may return fewer if augmenters fail.
#     # Each augmenter runs on the ORIGINAL text (not chained).
#     results = []
#     for aug in _CLASSICAL_AUGMENTERS:
#         out = _safe_augment(aug, text)
#         if out:
#             results.append(out)
#     return _deduplicate(results, text)


# # ─────────────────────────────────────────────────────────────────────────────
# # PATH B — GEMINI AUGMENTATION (complaint_length < 15)
# # ─────────────────────────────────────────────────────────────────────────────

# _GEMINI_PROMPT = (
#     "Rewrite the following civic complaint into {n} different natural-language "
#     "variations. Preserve the original meaning, sentiment, and complaint intent. "
#     "Keep the language informal and conversational -- do NOT make it overly formal.\n\n"
#     "Return ONLY a valid JSON array of strings and nothing else.\n\n"
#     "Example output format:\n"
#     '["variation 1", "variation 2", "variation 3"]\n\n'
#     'Complaint:\n"{text}"\n'
# )


# def gemini_augment(text: str, n: int = 2) -> list[str]:
#     # Call Gemini to generate up to n paraphrases of a short complaint.
#     # Returns an empty list on any failure so callers never crash.
#     text = str(text).strip()
#     if not text:
#         return []
#     try:
#         response = _gemini.generate_content(
#             _GEMINI_PROMPT.format(n=n, text=text)
#         )
#         # Handle multi-part responses (Gemini sometimes splits output)
#         try:
#             raw = response.text.strip()
#         except ValueError:
#             raw = "".join(
#                 part.text for part in response.candidates[0].content.parts
#             ).strip()
#         # Strip markdown code fences if present
#         if raw.startswith("```"):
#             raw = raw.split("```")[1]
#             if raw.startswith("json"):
#                 raw = raw[4:]
#             raw = raw.strip()

#         parsed = json.loads(raw)
#         if not isinstance(parsed, list):
#             return []

#         cleaned = [
#             str(item).strip() for item in parsed
#             if item and str(item).strip().lower() not in ("", "nan", text.lower())
#         ]
#         return _deduplicate(cleaned, text)

#     except Exception as e:
#         logging.warning(f"gemini_augment failed: {e}")
#         return []


# def gemini_augment_with_retry(
#     text:        str,
#     n:           int   = 3,
#     max_retries: int   = 3,
#     backoff:     float = 2.0,
# ) -> list[str]:
#     # gemini_augment with exponential back-off for rate-limit errors.
#     for attempt in range(1, max_retries + 1):
#         results = gemini_augment(text, n=n)
#         if results:
#             return results
#         wait = backoff ** attempt
#         logging.warning(
#             f"Gemini attempt {attempt}/{max_retries} failed. "
#             f"Retrying in {wait:.1f}s..."
#         )
#         time.sleep(wait)
#     return []


# # ─────────────────────────────────────────────────────────────────────────────
# # TIER FACTOR  (applies to BOTH paths)
# # ─────────────────────────────────────────────────────────────────────────────

# def _augment_factor(count: int) -> int:
#     if 1000 <= count <= 2000:
#         return 1
#     elif 500 <= count < 1000:
#         return 2
#     elif 1 <= count < 500:
#         return 3
#     else:
#         return 0  # Safe fallback if count > 2000


# # ─────────────────────────────────────────────────────────────────────────────
# # ROW-LEVEL CHECKPOINT HELPERS
# # ─────────────────────────────────────────────────────────────────────────────

# def _ckpt_load(ckpt_path: Path) -> tuple:
#     # Load a JSONL checkpoint file.
#     # Each line is a JSON record:
#     #   {"row_idx": int, "texts": [...], "labels": [...], "severities": [...]}
#     # Returns:
#     #   done_indices : set of int  -- row indices already processed
#     #   X_syn        : list[str]  -- synthetic texts accumulated so far
#     #   y_syn        : list[str]  -- corresponding labels
#     #   sev_syn      : list[str]  -- corresponding severities
#     done_indices = set()
#     X_syn, y_syn, sev_syn = [], [], []
#     if ckpt_path.exists():
#         with open(ckpt_path, "r", encoding="utf-8") as fh:
#             for line in fh:
#                 line = line.strip()
#                 if not line:
#                     continue
#                 try:
#                     rec = json.loads(line)
#                     done_indices.add(int(rec["row_idx"]))
#                     X_syn.extend(rec["texts"])
#                     y_syn.extend(rec["labels"])
#                     sev_syn.extend(rec["severities"])
#                 except (json.JSONDecodeError, KeyError):
#                     pass  # corrupt line -- skip silently, will be re-processed
#         print(
#             f"  [RESUME] Checkpoint loaded: {len(done_indices)} rows already done, "
#             f"{len(X_syn)} synthetic rows recovered."
#         )
#     return done_indices, X_syn, y_syn, sev_syn


# def _ckpt_append(ckpt_path: Path, row_idx: int,
#                  texts: list, labels: list, severities: list) -> None:
#     # Append one completed-row record to the JSONL checkpoint (one atomic line).
#     rec = {
#         "row_idx":    row_idx,
#         "texts":      texts,
#         "labels":     labels,
#         "severities": severities,
#     }
#     with open(ckpt_path, "a", encoding="utf-8") as fh:
#         fh.write(json.dumps(rec, ensure_ascii=False) + "\n")


# # ─────────────────────────────────────────────────────────────────────────────
# # MAIN AUGMENTATION ENTRY POINT  (with row-level resume support)
# # ─────────────────────────────────────────────────────────────────────────────

# def augment_dataset(
#     X:                     np.ndarray,
#     y:                     np.ndarray,
#     severity:              np.ndarray,
#     complaint_lengths:     np.ndarray,
#     checkpoint_path:       Path | None = None,
#     class_counts_override: dict | None = None,
# ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
#     # Augment a training set with optional row-level checkpoint / resume.
#     #
#     # Parameters
#     # ----------
#     # X                 : preprocessed complaint texts       shape (n,)
#     # y                 : class labels                       shape (n,)
#     # severity          : severity labels                    shape (n,)
#     # complaint_lengths : word counts per sample             shape (n,)
#     # checkpoint_path       : optional Path to a JSONL file.
#     # class_counts_override : optional dict {class_label: count}.
#     #   When provided, the augmentation factor is determined from
#     #   these counts instead of the local per-part counts.  Use this
#     #   when augmenting parts of a stratified split so the factor
#     #   reflects the FULL training-set distribution.
#     #   - If the file exists the function skips every row_idx already
#     #     written there and resumes from the next unprocessed row.
#     #   - After finishing each row the function appends a record so at
#     #     most one row of work is ever lost on a crash.
#     #   - Pass None to disable checkpointing entirely.
#     #
#     # Returns
#     # -------
#     # X_out, y_out, severity_out : augmented arrays (originals + synthetics)

#     # ── Load checkpoint (if any) ──────────────────────────────────────────────
#     if checkpoint_path is not None:
#         checkpoint_path = Path(checkpoint_path)
#         checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
#         done_indices, X_syn, y_syn, sev_syn = _ckpt_load(checkpoint_path)
#     else:
#         done_indices, X_syn, y_syn, sev_syn = set(), [], [], []

#     unique_classes, counts = np.unique(y, return_counts=True)

#     for cls, count in zip(unique_classes, counts):

#         factor_count = class_counts_override.get(cls, count) if class_counts_override else count
#         factor       = _augment_factor(int(factor_count))
#         cls_mask    = (y == cls)
#         cls_indices = np.where(cls_mask)[0]

#         # Only process rows not yet in the checkpoint
#         pending = [int(i) for i in cls_indices if int(i) not in done_indices]

#         long_mask  = complaint_lengths[cls_mask] > 15
#         short_mask = ~long_mask

#         print(
#             f"\n  Class '{cls}' | {count:,} samples | "
#             f"long>15: {long_mask.sum()} | short<=15: {short_mask.sum()} | "
#             f"factor: x{factor} | pending: {len(pending)}"
#         )

#         if factor == 0:
#             print(f"  [SKIP] Class '{cls}' has {count:,} samples (>2000) -- no augmentation needed.")
#             continue

#         if not pending:
#             print(f"  [OK] Class '{cls}' fully restored from checkpoint -- skipping.")
#             continue

#         with tqdm(
#             total      = len(pending) * factor,
#             desc       = f"  Class '{cls}'",
#             unit       = "pass",
#             colour     = "cyan",
#             leave      = True,
#             bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} passes "
#                          "[{elapsed}<{remaining}, {rate_fmt}]  "
#                          "syn_rows={postfix[0]}",
#             postfix    = [len(X_syn)],
#         ) as pbar:

#             for i in pending:
#                 is_long = complaint_lengths[i] > 15

#                 # Accumulate synthetic rows for THIS single original row
#                 row_texts:  list = []
#                 row_labels: list = []
#                 row_sevs:   list = []

#                 for _pass in range(factor):
#                     if is_long:
#                         new_texts = classical_augment(X[i])
#                     else:
#                         new_texts = gemini_augment_with_retry(X[i], n=3)

#                     for t in new_texts:
#                         t = preprocess(t)
#                         if t and t.strip():
#                             row_texts.append(t)
#                             row_labels.append(str(cls))
#                             row_sevs.append(str(severity[i]))

#                     pbar.update(1)

#                 # ── Accumulate in-memory lists ────────────────────────────────
#                 X_syn.extend(row_texts)
#                 y_syn.extend(row_labels)
#                 sev_syn.extend(row_sevs)
#                 pbar.postfix[0] = len(X_syn)

#                 # ── Write checkpoint line AFTER the row is fully done ─────────
#                 # If the process crashes between two rows, the checkpoint file
#                 # already has all previous rows safely on disk.
#                 if checkpoint_path is not None:
#                     _ckpt_append(
#                         checkpoint_path,
#                         row_idx    = i,
#                         texts      = row_texts,
#                         labels     = row_labels,
#                         severities = row_sevs,
#                     )

#     # ── Merge originals (always kept) + synthetics ────────────────────────────
#     X_merged   = list(X)        + X_syn
#     y_merged   = list(y)        + y_syn
#     sev_merged = list(severity) + sev_syn

#     # ── Final dedup / clean pass ──────────────────────────────────────────────
#     X_out, y_out, severity_out = _clean_df(X_merged, y_merged, sev_merged)

#     assert len(X_out) == len(y_out) == len(severity_out), \
#         "Length mismatch after augmentation"
#     assert all(isinstance(x, str) and x.strip() for x in X_out), \
#         "Empty or non-string texts found after augmentation"

#     n_orig  = len(X)
#     n_added = len(X_out) - n_orig
#     print(f"\n  Original : {n_orig:,}")
#     print(f"  Synthetic: {n_added:,}")
#     print(f"  Total    : {len(X_out):,}")

#     return X_out, y_out, severity_out

In [ ]:
# # ── Load environment variables from src/.env ──────────────────────────────────
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
# load_dotenv(dotenv_path=PROJECT_ROOT / "src" / ".env", override=True)

# vertexai.init(project="ai-grievance-sandi", location="us-central1")
# _gemini = GenerativeModel("gemini-2.5-flash")
# print("[OK] Gemini client initialised via Vertex AI (project: ai-grievance-sandi)")


# # ── Gemini Prompts ────────────────────────────────────────────────────────────
# _GEMINI_LONG_PROMPT = (
#     "Summarise the following civic complaint and write another variant of the complaint. "
#     "Preserve the original meaning, sentiment, and complaint intent. "
#     "Do NOT change or affect any severity metrics/scores mentioned or implied; keep the core issue identical.\n"
#     "Generate {n} different natural-language variations.\n\n"
#     "Return ONLY a valid JSON array of strings and nothing else.\n\n"
#     "Example output format:\n"
#     '["variation 1", "variation 2"]\n\n'
#     'Complaint:\n"{text}"\n'
# )

# _GEMINI_SHORT_PROMPT = (
#     "Perform sentence-level rewriting on the following civic complaint." 
#     "No two sentence-level rewritings should be the same."
#     "Use synonymous words for some words in the sentence based on the context, while preserving the original meaning, "
#     "sentiment, and complaint intent.\n"
#     "Generate {n} different natural-language variations.\n\n"
#     "Return ONLY a valid JSON array of strings and nothing else.\n\n"
#     "Example output format:\n"
#     '["variation 1", "variation 2"]\n\n'
#     'Complaint:\n"{text}"\n'
# )


# # ─────────────────────────────────────────────────────────────────────────────
# # HELPERS
# # ─────────────────────────────────────────────────────────────────────────────

# def _deduplicate(texts: list[str], original: str) -> list[str]:
#     # Remove duplicates and exact matches to the original (case-insensitive).
#     seen   = {original.lower()}
#     unique = []
#     for t in texts:
#         key = t.lower()
#         if key not in seen:
#             seen.add(key)
#             unique.append(t)
#     return unique


# def _clean_df(X: list, y: list, severity_score: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
#     frame = pd.DataFrame({"x": X, "y": y, "severity_score": severity_score}).dropna()
#     frame["x"] = frame["x"].astype(str).str.strip()
#     frame = frame[(frame["x"] != "") & (frame["x"].str.lower() != "nan")]
#     frame = frame.drop_duplicates()
#     return frame["x"].values, frame["y"].values, frame["severity_score"].values


# def _introduce_spelling_errors(text: str) -> str:
#     words = text.split()
#     if not words:
#         return text
    
#     num_words = len(words)
#     # Based on the length of the sentence, decide to introduce 2 or 3 spelling errors
#     num_errors = 3 if num_words > 50 else 1
    
#     # Select words suitable for spelling error injection (length > 2)
#     eligible_indices = [idx for idx, w in enumerate(words) if len(w) > 2]
#     if not eligible_indices:
#         eligible_indices = [idx for idx, w in enumerate(words) if len(w) > 1]
#         if not eligible_indices:
#             return text
            
#     num_errors = min(num_errors, len(eligible_indices))
#     selected_word_indices = np.random.choice(eligible_indices, size=num_errors, replace=False)
    
#     for idx in selected_word_indices:
#         word = words[idx]
#         w_list = list(word)
#         error_type = np.random.choice(['swap', 'replace', 'delete', 'insert'])
#         char_idx = np.random.randint(0, len(word))
        
#         if error_type == 'swap' and len(word) > 1:
#             swap_with = char_idx + 1 if char_idx < len(word) - 1 else char_idx - 1
#             w_list[char_idx], w_list[swap_with] = w_list[swap_with], w_list[char_idx]
#         elif error_type == 'delete' and len(word) > 2:
#             w_list.pop(char_idx)
#         elif error_type == 'replace':
#             random_char = chr(np.random.randint(97, 123))  # a-z
#             w_list[char_idx] = random_char
#         elif error_type == 'insert':
#             random_char = chr(np.random.randint(97, 123))
#             w_list.insert(char_idx, random_char)
            
#         words[idx] = "".join(w_list)
        
#     return " ".join(words)


# # ─────────────────────────────────────────────────────────────────────────────
# # LLM BASED AUGMENTATION ROUTING (via Vertex AI)
# # ─────────────────────────────────────────────────────────────────────────────

# def gemini_augment(text: str, is_long: bool, n: int = 2) -> list[str]:
#     text = str(text).strip()
#     if not text:
#         return []
    
#     prompt_template = _GEMINI_LONG_PROMPT if is_long else _GEMINI_SHORT_PROMPT
#     prompt = prompt_template.format(n=n, text=text)
    
#     try:
#         response = _gemini.generate_content(
#             prompt,
#             generation_config={
#                 "temperature": 0.7,
#                 "top_p": 0.95,
#                 "response_mime_type": "application/json",
#             }
#         )
#         try:
#             raw = response.text.strip()
#         except ValueError:
#             raw = "".join(
#                 part.text for part in response.candidates[0].content.parts
#             ).strip()
        
#         # Strip markdown code fences if present
#         if raw.startswith("```"):
#             raw = raw.split("```")[1]
#             if raw.startswith("json"):
#                 raw = raw[4:]
#             raw = raw.strip()

#         parsed = json.loads(raw)
#         if not isinstance(parsed, list):
#             return []

#         cleaned = [
#             str(item).strip() for item in parsed
#             if item and str(item).strip().lower() not in ("", "nan", text.lower())
#         ]
#         return _deduplicate(cleaned, text)

#     except Exception as e:
#         logging.warning(f"gemini_augment failed: {e}")
#         return []


# def gemini_augment_with_retry(
#     text:        str,
#     is_long:     bool,
#     n:           int   = 3,
#     max_retries: int   = 3,
#     backoff:     float = 2.0,
# ) -> list[str]:
#     # gemini_augment with exponential back-off for rate-limit errors.
#     for attempt in range(1, max_retries + 1):
#         results = gemini_augment(text, is_long=is_long, n=n)
#         if results:
#             return results
#         wait = backoff ** attempt
#         logging.warning(
#             f"Gemini attempt {attempt}/{max_retries} failed. "
#             f"Retrying in {wait:.1f}s..."
#         )
#         time.sleep(wait)
#     return []


# # ─────────────────────────────────────────────────────────────────────────────
# # TIER FACTOR
# # ─────────────────────────────────────────────────────────────────────────────

# def _augment_factor(count: int) -> int:
#     if 1000 <= count <= 2000:
#         return 1
#     elif 500 <= count < 1000:
#         return 2
#     elif 100 <= count < 500:
#         return 3
#     elif 1 <= count < 100:
#         return 5
#     else:
#         return 0


# # ─────────────────────────────────────────────────────────────────────────────
# # ROW-LEVEL CHECKPOINT HELPERS
# # ─────────────────────────────────────────────────────────────────────────────

# def _ckpt_load(ckpt_path: Path) -> tuple:
#     done_indices = set()
#     X_syn, y_syn, sev_syn = [], [], []
#     if ckpt_path.exists():
#         with open(ckpt_path, "r", encoding="utf-8") as fh:
#             for line in fh:
#                 line = line.strip()
#                 if not line:
#                     continue
#                 try:
#                     rec = json.loads(line)
#                     done_indices.add(int(rec["row_idx"]))
#                     X_syn.extend(rec["texts"])
#                     y_syn.extend(rec["labels"])
#                     sev_syn.extend(rec.get("severity_scores", rec.get("severities", [])))
#                 except (json.JSONDecodeError, KeyError):
#                     pass
#         print(
#             f"  [RESUME] Checkpoint loaded: {len(done_indices)} rows already done, "
#             f"{len(X_syn)} synthetic rows recovered."
#         )
#     return done_indices, X_syn, y_syn, sev_syn


# def _ckpt_append(ckpt_path: Path, row_idx: int,
#                  texts: list, labels: list, severity_scores: list) -> None:
#     rec = {
#         "row_idx":         row_idx,
#         "texts":           texts,
#         "labels":          labels,
#         "severity_scores": severity_scores,
#     }
#     with open(ckpt_path, "a", encoding="utf-8") as fh:
#         fh.write(json.dumps(rec, ensure_ascii=False) + "\n")


# # ─────────────────────────────────────────────────────────────────────────────
# # MAIN AUGMENTATION ENTRY POINT
# # ─────────────────────────────────────────────────────────────────────────────

# def augment_dataset(
#     X:                     np.ndarray,
#     y:                     np.ndarray,
#     severity_score:        np.ndarray,
#     checkpoint_path:       Path | None = None,
#     class_counts_override: dict | None = None,
#     preprocess_fn:         callable = None,
# ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    
#     # ── Load checkpoint (if any) ──────────────────────────────────────────────
#     if checkpoint_path is not None:
#         checkpoint_path = Path(checkpoint_path)
#         checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
#         done_indices, X_syn, y_syn, sev_syn = _ckpt_load(checkpoint_path)
#     else:
#         done_indices, X_syn, y_syn, sev_syn = set(), [], [], []

#     unique_classes, counts = np.unique(y, return_counts=True)

#     for cls, count in zip(unique_classes, counts):
#         factor_count = class_counts_override.get(cls, count) if class_counts_override else count
#         factor       = _augment_factor(int(factor_count))
#         cls_mask     = (y == cls)
#         cls_indices  = np.where(cls_mask)[0]

#         pending = [int(i) for i in cls_indices if int(i) not in done_indices]

#         cls_lengths = np.array([len(str(t).split()) for t in X[cls_mask]])
#         long_mask   = cls_lengths > 110
#         short_mask = ~long_mask

#         print(
#             f"\n  Class '{cls}' | {count:,} samples | "
#             f"long>110: {long_mask.sum()} | short<=110: {short_mask.sum()} | "
#             f"factor: x{factor} | pending: {len(pending)}"
#         )

#         if factor == 0:
#             print(f"  [SKIP] Class '{cls}' has {count:,} samples (>2000) -- no augmentation needed.")
#             continue

#         if not pending:
#             print(f"  [OK] Class '{cls}' fully restored from checkpoint -- skipping.")
#             continue

#         with tqdm(
#             total      = len(pending),
#             desc       = f"  Class '{cls}'",
#             unit       = "row",
#             colour     = "cyan",
#             leave      = True,
#             bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} rows "
#                          "[{elapsed}<{remaining}, {rate_fmt}]  "
#                          "syn_rows={postfix[0]}",
#             postfix    = [len(X_syn)],
#         ) as pbar:

#             for i in pending:
#                 complaint_len = len(str(X[i]).split())
#                 is_long = complaint_len > 110

#                 row_texts:  list = []
#                 row_labels: list = []
#                 row_sevs:   list = []

#                 # Request factor * 2 options in 1 API call to handle filtering / duplicates efficiently
#                 new_texts = gemini_augment_with_retry(X[i], is_long=is_long, n=factor * 2)
#                 new_texts = new_texts[:factor]

#                 for t in new_texts:
#                     if preprocess_fn is not None:
#                         t = preprocess_fn(t)
#                     if t and t.strip():
#                         row_texts.append(t)
#                         row_labels.append(str(cls))
#                         row_sevs.append(str(severity_score[i]))

#                 pbar.update(1)

#                 # ── Accumulate in-memory lists ────────────────────────────────
#                 X_syn.extend(row_texts)
#                 y_syn.extend(row_labels)
#                 sev_syn.extend(row_sevs)
#                 pbar.postfix[0] = len(X_syn)

#                 # ── Write checkpoint line AFTER the row is fully done ─────────
#                 if checkpoint_path is not None:
#                     _ckpt_append(
#                         checkpoint_path,
#                         row_idx         = i,
#                         texts           = row_texts,
#                         labels          = row_labels,
#                         severity_scores = row_sevs,
#                     )

#                 # Polite rate-limiting delay between requests
#                 time.sleep(1.0)

#     # ── Step 3: Randomly select ~5% of synthetic complaints to introduce spelling errors ──
#     # Done only on X_syn to guarantee that the original rows are kept exactly preserved.
#     if len(X_syn) > 0:
#         num_to_select = int(round(0.05 * len(X_syn)))
#         if num_to_select > 0:
#             # Weighted probability: longer sentences (by word count) have a higher chance of selection
#             syn_lengths = np.array([len(x.split()) for x in X_syn])
#             total_len = syn_lengths.sum()
#             if total_len > 0:
#                 probs = syn_lengths / total_len
#             else:
#                 probs = np.ones(len(X_syn)) / len(X_syn)
            
#             selected_indices = np.random.choice(len(X_syn), size=num_to_select, replace=False, p=probs)
#             for idx in selected_indices:
#                 X_syn[idx] = _introduce_spelling_errors(X_syn[idx])

#     # ── Merge originals (always kept) + synthetics ────────────────────────────
#     X_merged   = list(X)             + X_syn
#     y_merged   = list(y)             + y_syn
#     sev_merged = list(severity_score) + sev_syn

#     # ── Final dedup / clean pass ──────────────────────────────────────────────
#     X_out, y_out, severity_out = _clean_df(X_merged, y_merged, sev_merged)

#     assert len(X_out) == len(y_out) == len(severity_out), \
#         "Length mismatch after augmentation"
#     assert all(isinstance(x, str) and x.strip() for x in X_out), \
#         "Empty or non-string texts found after augmentation"

#     n_orig  = len(X)
#     n_added = len(X_out) - n_orig
#     print(f"\n  Original : {n_orig:,}")
#     print(f"  Synthetic: {n_added:,}")
#     print(f"  Total    : {len(X_out):,}")

#     return X_out, y_out, severity_out


In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # 5-FOLD STRATIFIED CROSS-VALIDATION -- Civic Agency Classification
# # ─────────────────────────────────────────────────────────────────────────────
# #
# # Workflow:
# #
# # 1. Create two preprocessed datasets from the original dataset:
# #    - Classical ML Dataset     → preprocess_classical_ml()
# #    - BERT NLP Dataset         → preprocess_nlp_bert()
# #
# # 2. Split each dataset into 5 stratified folds.
# #
# # 3. Augment each fold independently and only once:
# #    - Classical ML folds → df_classical_aug_1 ... df_classical_aug_5
# #    - BERT NLP folds     → df_bert_aug_1 ... df_bert_aug_5
# #
# # 4. For each CV iteration:
# #    - Training   = 4 augmented folds
# #    - Validation = 1 original (non-augmented) fold
# #
# # 5. Repeat Step 4 for all 5 folds.
# #
# # 6. Save the fold-wise train/validation splits as:
# #    - df_classical_ml_cv.joblib
# #    - df_bert_nlp_cv.joblib
# #
# # Result:
# #    - Classical ML CV Dataset
# #      (preprocess_classical_ml + augmentation)
# #
# #    - BERT NLP CV Dataset
# #      (preprocess_nlp_bert + augmentation)
# # ─────────────────────────────────────────────────────────────────────────────

# # Exact save paths from Step 6
# SAVE_PATH_CLASSICAL = PROJECT_ROOT / "data" / "processed" / "df_classical_ml_cv.joblib"
# SAVE_PATH_BERT      = PROJECT_ROOT / "data" / "processed" / "df_bert_nlp_cv.joblib"

# SAVE_PATH_CLASSICAL.parent.mkdir(parents=True, exist_ok=True)
# SAVE_PATH_BERT.parent.mkdir(parents=True, exist_ok=True)

# CKPT_DIR  = PROJECT_ROOT / "data" / "processed" / "aug_checkpoints_v2"
# PARTS_DIR = PROJECT_ROOT / "data" / "processed" / "aug_parts_df_v2"
# CKPT_DIR.mkdir(parents=True, exist_ok=True)
# PARTS_DIR.mkdir(parents=True, exist_ok=True)

# if SAVE_PATH_CLASSICAL.exists() and SAVE_PATH_BERT.exists():
#     print(f"[OK] Loading cached CV splits:")
#     fold_data_list_classical = joblib.load(SAVE_PATH_CLASSICAL)
#     fold_data_list_bert      = joblib.load(SAVE_PATH_BERT)
#     print(f"   Loaded Classical ML CV Dataset: {len(fold_data_list_classical)} folds.")
#     print(f"   Loaded BERT NLP CV Dataset: {len(fold_data_list_bert)} folds.")

# else:
#     print("[INFO] Running 5-Fold Stratified CV Pipeline...\n")

#     # Assuming dataframe `df` is available. 
#     # NOTE: Using "description" and "civic_agency_title" if your column names differ!
#     y_arr = df["civic_agency_title"].values
#     severity_arr = df["severity_score"].values

#     _cls_unique, _cls_counts = np.unique(y_arr, return_counts=True)
#     full_class_counts = dict(zip(_cls_unique, _cls_counts.astype(int)))

#     # =========================================================================
#     # Step 1: Create two preprocessed datasets from the original dataset
#     # =========================================================================
#     print("Step 1: Creating preprocessed datasets...")
#     X_classical = np.array([preprocess_classical_ml(str(t)) for t in df["description"].values])
#     X_bert      = np.array([preprocess_nlp_bert(str(t))     for t in df["description"].values])

#     # =========================================================================
#     # Step 2: Split each dataset into 5 stratified folds
#     # =========================================================================
#     print("Step 2: Splitting into 5 stratified folds...")
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
#     # The split indices are identical for both datasets since they share `y_arr`
#     all_splits = list(skf.split(np.zeros(len(y_arr)), y_arr))

#     # =========================================================================
#     # Step 3: Augment each fold independently and only once
#     # =========================================================================
#     print("Step 3: Augmenting each fold independently...")
    
#     df_classical_aug_parts = []
#     df_bert_aug_parts      = []
#     original_parts         = []

#     for part_idx in range(5):
#         _, part_indices = all_splits[part_idx]

#         # Extract fold slices
#         part_y    = y_arr[part_indices]
#         part_sev  = severity_arr[part_indices]
        
#         part_X_classical = X_classical[part_indices]
#         part_X_bert      = X_bert[part_indices]

#         # Save the original un-augmented slices for the validation sets later
#         original_parts.append({
#             "X_classical":    part_X_classical,
#             "X_bert":         part_X_bert,
#             "y":              part_y,
#             "severity_score": part_sev,
#             "idx":            part_indices
#         })

#         # --- Augment Classical ML Fold ---
#         part_save_classical = PARTS_DIR / f"df_classical_aug_{part_idx+1}.joblib"
#         if part_save_classical.exists():
#             aug_classical = joblib.load(part_save_classical)
#         else:
#             print(f"\n--- Augmenting Classical ML Fold {part_idx+1} ---")
#             part_ckpt_classical = CKPT_DIR / f"classical_part_{part_idx+1}.jsonl"
#             aug_X_class, aug_y_class, aug_sev_class = augment_dataset(
#                 X                     = part_X_classical,
#                 y                     = part_y,
#                 severity_score        = part_sev,
#                 checkpoint_path       = part_ckpt_classical,
#                 preprocess_fn         = preprocess_classical_ml,
#                 class_counts_override = full_class_counts,
#             )
#             aug_classical = {"X": aug_X_class, "y": aug_y_class, "severity_score": aug_sev_class}
#             joblib.dump(aug_classical, part_save_classical, compress=3)
#             if part_ckpt_classical.exists(): part_ckpt_classical.unlink()
        
#         df_classical_aug_parts.append(aug_classical)

#         # --- Augment BERT NLP Fold ---
#         part_save_bert = PARTS_DIR / f"df_bert_aug_{part_idx+1}.joblib"
#         if part_save_bert.exists():
#             aug_bert = joblib.load(part_save_bert)
#         else:
#             print(f"\n--- Augmenting BERT NLP Fold {part_idx+1} ---")
#             part_ckpt_bert = CKPT_DIR / f"bert_part_{part_idx+1}.jsonl"
#             aug_X_bert, aug_y_bert, aug_sev_bert = augment_dataset(
#                 X                     = part_X_bert,
#                 y                     = part_y,
#                 severity_score        = part_sev,
#                 checkpoint_path       = part_ckpt_bert,
#                 preprocess_fn         = preprocess_nlp_bert,
#                 class_counts_override = full_class_counts,
#             )
#             aug_bert = {"X": aug_X_bert, "y": aug_y_bert, "severity_score": aug_sev_bert}
#             joblib.dump(aug_bert, part_save_bert, compress=3)
#             if part_ckpt_bert.exists(): part_ckpt_bert.unlink()
            
#         df_bert_aug_parts.append(aug_bert)

#     # =========================================================================
#     # Step 4 & 5: For each CV iteration, assemble Training (4) & Validation (1)
#     # =========================================================================
#     print("\nSteps 4 & 5: Assembling CV iterations...")

#     df_classical_ml_cv = []
#     df_bert_nlp_cv     = []

#     for fold_idx in range(5):
#         # Build Classical Training Set (4 augmented folds)
#         train_X_class_parts   = [df_classical_aug_parts[i]["X"]              for i in range(5) if i != fold_idx]
#         train_y_class_parts   = [df_classical_aug_parts[i]["y"]              for i in range(5) if i != fold_idx]
#         train_sev_class_parts = [df_classical_aug_parts[i]["severity_score"] for i in range(5) if i != fold_idx]

#         # Build BERT Training Set (4 augmented folds)
#         train_X_bert_parts   = [df_bert_aug_parts[i]["X"]              for i in range(5) if i != fold_idx]
#         train_y_bert_parts   = [df_bert_aug_parts[i]["y"]              for i in range(5) if i != fold_idx]
#         train_sev_bert_parts = [df_bert_aug_parts[i]["severity_score"] for i in range(5) if i != fold_idx]

#         # Assemble Classical Dict
#         df_classical_ml_cv.append({
#             "train_X":              np.concatenate(train_X_class_parts),
#             "train_y":              np.concatenate(train_y_class_parts),
#             "train_severity_score": np.concatenate(train_sev_class_parts),
#             "val_X":                original_parts[fold_idx]["X_classical"],
#             "val_y":                original_parts[fold_idx]["y"],
#             "val_severity_score":   original_parts[fold_idx]["severity_score"],
#             "val_idx":              original_parts[fold_idx]["idx"],
#         })

#         # Assemble BERT Dict
#         df_bert_nlp_cv.append({
#             "train_X":              np.concatenate(train_X_bert_parts),
#             "train_y":              np.concatenate(train_y_bert_parts),
#             "train_severity_score": np.concatenate(train_sev_bert_parts),
#             "val_X":                original_parts[fold_idx]["X_bert"],
#             "val_y":                original_parts[fold_idx]["y"],
#             "val_severity_score":   original_parts[fold_idx]["severity_score"],
#             "val_idx":              original_parts[fold_idx]["idx"],
#         })

#         print(f"  CV Iteration {fold_idx+1}/5 assembled.")

#     # =========================================================================
#     # Step 6: Save the fold-wise train/validation splits
#     # =========================================================================
#     joblib.dump(df_classical_ml_cv, SAVE_PATH_CLASSICAL, compress=3)
#     print(f"\nStep 6 [OK] Saved -> {SAVE_PATH_CLASSICAL.name}")

#     joblib.dump(df_bert_nlp_cv, SAVE_PATH_BERT, compress=3)
#     print(f"Step 6 [OK] Saved -> {SAVE_PATH_BERT.name}")

### 3.6  Analysis of the synthetic data

This part analyses the synthetic joblib data.

In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # Clean Augmented Data (with severity_reason + complaint_length)
# # ─────────────────────────────────────────────────────────────────────────────

# CLEAN_MODES = [
#     {
#         "aug_path": PROJECT_ROOT / "data" / "processed" / "df_classical_ml_cv.joblib",
#         "final_path": PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final.joblib",
#         "name": "Classical ML"
#     },
#     {
#         "aug_path": PROJECT_ROOT / "data" / "processed" / "df_bert_nlp_cv.joblib",
#         "final_path": PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final_bert.joblib",
#         "name": "BERT NLP"
#     }
# ]

# def clean_fold_train(X, y, severity, reason):
#     """Remove NaN, blank, duplicate, and < 3 word samples from training arrays."""
#     df_temp = pd.DataFrame({"X": X, "y": y, "severity": severity, "reason": reason})
#     before = len(df_temp)

#     # 1. Remove NaN / blank
#     mask_valid = df_temp["X"].apply(
#         lambda x: isinstance(x, str) and x.strip() != "" and x.strip().lower() != "nan"
#     )
#     df_temp = df_temp[mask_valid]
#     dropped_nan = before - len(df_temp)

#     # 2. Remove duplicates (keep first occurrence)
#     before_dup = len(df_temp)
#     df_temp = df_temp.drop_duplicates(subset=["X"], keep="first")
#     dropped_dup = before_dup - len(df_temp)

#     # 3. Remove < 3 word complaints
#     before_short = len(df_temp)
#     df_temp = df_temp[df_temp["X"].apply(lambda x: len(x.split()) >= 3)]
#     dropped_short = before_short - len(df_temp)

#     # 4. Compute complaint_length (word count)
#     complaint_length = df_temp["X"].apply(lambda x: len(x.split())).values

#     print(f"    Removed -> NaN/blank: {dropped_nan}, duplicates: {dropped_dup}, "
#           f"< 3 words: {dropped_short}  |  {before:,} -> {len(df_temp):,}")

#     return (df_temp["X"].values, df_temp["y"].values,
#             df_temp["severity"].values, df_temp["reason"].values,
#             complaint_length)

# for cfg in CLEAN_MODES:
#     print(f"\n============================================================")
#     print(f"  CLEANING AUGMENTED DATA FOR: {cfg['name']}")
#     print(f"============================================================")
    
#     if not cfg["aug_path"].exists():
#         print(f"[WARNING] Augmented path does not exist, skipping: {cfg['aug_path']}")
#         continue
        
#     _aug_data = joblib.load(cfg["aug_path"])
#     fold_data_clean = []

#     for fi, fold in enumerate(_aug_data):
#         print(f"  Fold {fi}:")
#         clean_X, clean_y, clean_sev, clean_reason, clean_len = clean_fold_train(
#             fold["train_X"], fold["train_y"],
#             fold["train_severity_score"], fold["train_severity_reason"]
#         )

#         # Compute complaint_length for validation set too
#         val_len = np.array([len(str(x).split()) for x in fold["val_X"]])

#         fold_data_clean.append({
#             "train_X":                clean_X,
#             "train_y":                clean_y,
#             "train_severity_score":   clean_sev,
#             "train_severity_reason":  clean_reason,
#             "train_complaint_length": clean_len,
#             "val_X":                  fold["val_X"],
#             "val_y":                  fold["val_y"],
#             "val_severity_score":     fold["val_severity_score"],
#             "val_severity_reason":    fold["val_severity_reason"],
#             "val_complaint_length":   val_len,
#             "val_idx":                fold["val_idx"],
#         })

#     joblib.dump(fold_data_clean, cfg["final_path"], compress=3)
#     print(f"\n  [OK] Saved cleaned folds -> {cfg['final_path'].name}")
#     print(f"       Keys: {list(fold_data_clean[0].keys())}")

### 3.7  Shared Utilities for Grid Search

A reusable `run_grid_search()` helper runs cross-validated hyperparameter tuning for any scikit-learn pipeline, saves the best params, OOF confusion matrix, and metrics JSON — keeping the per-model cells concise.


In [ ]:
SEED = 43
random.seed(SEED); np.random.seed(SEED)

# ── Load civic-agency fold data ──────────────────────────────────────────────
CIVIC_OUTPUT = CHARTS_DIR / "civic_agency_results"
fold_data_path = PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final.joblib"
fold_data = joblib.load(fold_data_path)
print("✅ Civic agency fold data loaded.")

# 1. Create an empty array the size of the full dataset (or sum of val sizes)
# Note: Since val_idx refers to indices in y_train, we should size it to len(y_train)
total_size = sum(len(d["val_y"]) for d in fold_data)
y_global = np.empty(total_size, dtype=object)
# 2. Fill it using the original indices
for d in fold_data:
    y_global[d["val_idx"]] = d["val_y"]
# 3. Now y_global and oof use the same index system
labels = sorted(np.unique(y_global))
print(f"Classes ({len(labels)}): {labels}")


def _next_version(path: Path, base_name: str) -> int:
    """Return the next unused version index for output files."""
    i = 1
    while (path / f"{base_name}_{i}.json").exists():
        i += 1
    return i

def run_grid_search(model_name, build_fn, param_grid, fold_data, y_global, labels, output_root):
    """
    Cross-validated grid search over param_grid.
    Saves best params, OOF confusion matrix PNG, and metrics JSON.
    Returns (best_params, best_metrics).
    """
    SEED= 43
    out_dir = output_root / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    version = _next_version(out_dir, f"results_{model_name}")
    best_f1, best_params, best_metrics = -1, None, None
    print(f"\n{'='*55}\n  GRID SEARCH — {model_name.upper()}\n{'='*55}")
    for params in ParameterGrid(param_grid):

        random.seed(SEED)
        np.random.seed(SEED)

        # Lists to store metrics per fold
        train_accs, train_precs, train_recs = [], [], []
        train_f1_macros, train_f1_micros, train_f1_weighteds = [], [], []
        
        val_accs, val_precs, val_recs = [], [], []
        val_f1_macros, val_f1_micros, val_f1_weighteds = [], [], []
        for d in fold_data:
            m = build_fn()
            random.seed(SEED)
            np.random.seed(SEED)
            m.set_params(**params)
            m.fit(d["train_X"], d["train_y"])
            
            # ── Train metrics ──
            train_preds = m.predict(d["train_X"])
            train_accs.append(accuracy_score(d["train_y"], train_preds))
            train_precs.append(precision_score(d["train_y"], train_preds, average="macro", zero_division=0))
            train_recs.append(recall_score(d["train_y"], train_preds, average="macro", zero_division=0))
            train_f1_macros.append(f1_score(d["train_y"], train_preds, average="macro"))
            train_f1_micros.append(f1_score(d["train_y"], train_preds, average="micro"))
            train_f1_weighteds.append(f1_score(d["train_y"], train_preds, average="weighted"))
            
            # ── Val metrics ──
            val_preds = m.predict(d["val_X"])
            val_accs.append(accuracy_score(d["val_y"], val_preds))
            val_precs.append(precision_score(d["val_y"], val_preds, average="macro", zero_division=0))
            val_recs.append(recall_score(d["val_y"], val_preds, average="macro", zero_division=0))
            val_f1_macros.append(f1_score(d["val_y"], val_preds, average="macro"))
            val_f1_micros.append(f1_score(d["val_y"], val_preds, average="micro"))
            val_f1_weighteds.append(f1_score(d["val_y"], val_preds, average="weighted"))
        # Averages
        avg_val_f1 = float(np.mean(val_f1_macros))
        print(f"  {params}")
        print(f"    Val  | F1-macro: {avg_val_f1:.4f} | F1-micro: {np.mean(val_f1_micros):.4f} | F1-wgt: {np.mean(val_f1_weighteds):.4f} | Acc: {np.mean(val_accs):.4f} | P: {np.mean(val_precs):.4f} | R: {np.mean(val_recs):.4f}")
        print(f"    Train| F1-macro: {np.mean(train_f1_macros):.4f} | F1-micro: {np.mean(train_f1_micros):.4f} | F1-wgt: {np.mean(train_f1_weighteds):.4f} | Acc: {np.mean(train_accs):.4f} | P: {np.mean(train_precs):.4f} | R: {np.mean(train_recs):.4f}")
        
        if avg_val_f1 > best_f1:
            best_f1 = avg_val_f1
            best_params = params
            best_metrics = {
                "val_f1_macro": float(np.mean(val_f1_macros)),
                "val_f1_micro": float(np.mean(val_f1_micros)),
                "val_f1_weighted": float(np.mean(val_f1_weighteds)),
                "val_accuracy": float(np.mean(val_accs)),
                "val_precision": float(np.mean(val_precs)),
                "val_recall": float(np.mean(val_recs)),
                "train_f1_macro": float(np.mean(train_f1_macros)),
                "train_f1_micro": float(np.mean(train_f1_micros)),
                "train_f1_weighted": float(np.mean(train_f1_weighteds)),
                "train_accuracy": float(np.mean(train_accs)),
                "train_precision": float(np.mean(train_precs)),
                "train_recall": float(np.mean(train_recs)),
            }
    print(f"\n✅ Best Val F1-macro: {best_f1:.4f}  |  Params: {best_params}")
    # OOF confusion matrix
    oof = np.empty_like(y_global, dtype=object)
    for d in fold_data:
        m = build_fn(); m.set_params(**best_params)
        m.fit(d["train_X"], d["train_y"])
        oof[d["val_idx"]] = m.predict(d["val_X"])
    cm = confusion_matrix(y_global, oof, labels=labels)
    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar=False)
    plt.title(f"Average Metrics Confusion Matrix — {model_name} (v{version})")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = out_dir / f"3.7_grid_search_confusion_matrix_{model_name}_{version}.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    results = {
        "version": version, "model": model_name,
        "best_params": {k: list(v) if isinstance(v, tuple) else v for k, v in best_params.items()},
        "metrics": best_metrics,
    }
    with open(out_dir / f"grid_search_results_{model_name}_{version}.json", "w") as fh:
        json.dump(results, fh, indent=4)
    return best_params, best_metrics

### 3.6c  Load cleaned data (Unused - merged into cleaning logic)


### 3.7  Logistic Regression — Civic Agency


In [ ]:
# lr_params, lr_metrics = run_grid_search(
#     model_name="logistic_regression",
#     build_fn=lambda: Pipeline([
#         ("tfidf", TfidfVectorizer(sublinear_tf=True, min_df=3)),
#         ("clf",   LogisticRegression(solver="lbfgs", class_weight="balanced",
#                                      max_iter=2000, tol=1e-3, n_jobs=8, random_state=SEED)),
#     ]),
#     param_grid={
#         "tfidf__max_features": [3000, 5000, 8000, 10000, 12000],
#         "tfidf__ngram_range":  [(1, 1), (1, 2), (1, 3)],
#         "clf__C":              [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
#     },
#     fold_data=fold_data, y_global=y_global,
#     labels=labels, output_root=CIVIC_OUTPUT,
# )

#### Complaint-Length vs Misclassification

We test statistically whether shorter complaints are harder to classify correctly. This guides future feature engineering (e.g. minimum-length filters, length-aware models).


In [ ]:
# all_texts, all_true, all_pred = [], [], []

# for d in fold_data:
#     m = Pipeline([
#         ("tfidf", TfidfVectorizer(sublinear_tf=True, min_df=3)),
#         ("clf",   LogisticRegression(solver="lbfgs", class_weight="balanced",
#                                      max_iter=2000, tol=1e-3, n_jobs=-1, random_state=SEED)),
#     ])
#     m.set_params(**lr_params)
#     m.fit(d["train_X"], d["train_y"])
#     preds = m.predict(d["val_X"])
#     all_texts.extend(d["val_X"]); all_true.extend(d["val_y"]); all_pred.extend(preds)

# analysis_df = pd.DataFrame({
#     "text": all_texts, "true_label": all_true, "pred_label": all_pred,
#     "complaint_length": [len(str(t).split()) for t in all_texts],
# })
# analysis_df["correct"] = analysis_df["true_label"] == analysis_df["pred_label"]

# correct_len = analysis_df.loc[ analysis_df["correct"], "complaint_length"]
# miscls_len  = analysis_df.loc[~analysis_df["correct"], "complaint_length"]

# print(f"Correctly classified  : {len(correct_len):,}  | mean: {correct_len.mean():.1f} words")
# print(f"Misclassified         : {len(miscls_len):,}  | mean: {miscls_len.mean():.1f} words")

# _, t_p = ttest_ind(correct_len, miscls_len, equal_var=False)
# _, u_p = mannwhitneyu(correct_len, miscls_len, alternative="two-sided")
# n1, n2 = len(correct_len), len(miscls_len)
# pooled_std = np.sqrt(((n1-1)*correct_len.std()**2 + (n2-1)*miscls_len.std()**2) / (n1+n2-2))
# cohens_d   = (miscls_len.mean() - correct_len.mean()) / (pooled_std + 1e-9)

# size = ("Negligible" if abs(cohens_d) < 0.2 else
#         "Small"      if abs(cohens_d) < 0.5 else
#         "Medium"     if abs(cohens_d) < 0.8 else "Large")

# print(f"\nMann-Whitney p = {u_p:.2e}  |  Cohen's d = {cohens_d:.3f} ({size})")
# print(f"Result: Statistically significant (p={u_p:.2e}) but effect size is {size.lower()} "
#       f"(d={cohens_d:.3f}) — complaint length has minimal practical impact on prediction accuracy.")


### 3.8  LinearSVC — Civic Agency


In [ ]:
# svc_params, svc_metrics = run_grid_search(
#     model_name="linearsvc",
#     build_fn=lambda: Pipeline([
#         ("tfidf", TfidfVectorizer()),
#         ("clf",   LinearSVC(penalty="l2", multi_class="ovr", class_weight="balanced",
#                             max_iter=2000, dual="auto", random_state=SEED)),
#     ]),
#     param_grid={
#         "tfidf__max_features": [1_000, 3_000, 5_000, 8_000, 10_000, 12_000],
#         "tfidf__ngram_range":  [(1, 1), (1, 2), (1,3)],
#         "clf__C":              [0.1, 0.5, 1.0, 2.0, 3.0, 5.0],
#     },
#     fold_data=fold_data, y_global=y_global,
#     labels=labels, output_root=CIVIC_OUTPUT,
# )

In [ ]:
# all_texts, all_true, all_pred = [], [], []

# for d in fold_data:
#     m = Pipeline([
#         ("tfidf", TfidfVectorizer()),
#         ("clf",   LinearSVC(penalty="l2", multi_class="ovr", class_weight="balanced",
#                             max_iter=2000, dual="auto", random_state=SEED)),
#     ])
#     m.set_params(**svc_params)
#     m.fit(d["train_X"], d["train_y"])
#     preds = m.predict(d["val_X"])
#     all_texts.extend(d["val_X"]); all_true.extend(d["val_y"]); all_pred.extend(preds)

# analysis_df = pd.DataFrame({
#     "text": all_texts, "true_label": all_true, "pred_label": all_pred,
#     "complaint_length": [len(str(t).split()) for t in all_texts],
# })
# analysis_df["correct"] = analysis_df["true_label"] == analysis_df["pred_label"]

# correct_len = analysis_df.loc[ analysis_df["correct"], "complaint_length"]
# miscls_len  = analysis_df.loc[~analysis_df["correct"], "complaint_length"]

# print(f"Correctly classified  : {len(correct_len):,}  | mean: {correct_len.mean():.1f} words")
# print(f"Misclassified         : {len(miscls_len):,}  | mean: {miscls_len.mean():.1f} words")

# _, t_p = ttest_ind(correct_len, miscls_len, equal_var=False)
# _, u_p = mannwhitneyu(correct_len, miscls_len, alternative="two-sided")
# n1, n2 = len(correct_len), len(miscls_len)
# pooled_std = np.sqrt(((n1-1)*correct_len.std()**2 + (n2-1)*miscls_len.std()**2) / (n1+n2-2))
# cohens_d   = (miscls_len.mean() - correct_len.mean()) / (pooled_std + 1e-9)

# size = ("Negligible" if abs(cohens_d) < 0.2 else
#         "Small"      if abs(cohens_d) < 0.5 else
#         "Medium"     if abs(cohens_d) < 0.8 else "Large")

# print(f"\nMann-Whitney p = {u_p:.2e}  |  Cohen's d = {cohens_d:.3f} ({size})")
# print(f"Result: Statistically significant (p={u_p:.2e}) but effect size is {size.lower()} "
#       f"(d={cohens_d:.3f}) — complaint length has minimal practical impact on prediction accuracy.")

### 3.9  MultinomialNB — Civic Agency


In [ ]:
# nb_params, nb_metrics = run_grid_search(
#     model_name="multinomial_nb",
#     build_fn=lambda: Pipeline([
#         ("tfidf", TfidfVectorizer(sublinear_tf=True)),
#         ("clf",   MultinomialNB()),
#     ]),
#     param_grid={
#         "tfidf__max_features": [5_000, 8_000, 10_000, 12_000],
#         "tfidf__ngram_range":  [(1, 1), (1, 2), (1, 3)],
#         "clf__alpha":          [0.01, 0.1, 0.5, 1.0, 2.0],
#         "clf__fit_prior":      [True, False],
#     },
#     fold_data=fold_data, y_global=y_global,
#     labels=labels, output_root=CIVIC_OUTPUT,
# )

In [ ]:
# all_texts, all_true, all_pred = [], [], []

# for d in fold_data:
#     m = Pipeline([
#         ("tfidf", TfidfVectorizer(sublinear_tf=True)),
#         ("clf",   MultinomialNB()),
#     ])
#     m.set_params(**nb_params)
#     m.fit(d["train_X"], d["train_y"])
#     preds = m.predict(d["val_X"])
#     all_texts.extend(d["val_X"]); all_true.extend(d["val_y"]); all_pred.extend(preds)

# analysis_df = pd.DataFrame({
#     "text": all_texts, "true_label": all_true, "pred_label": all_pred,
#     "complaint_length": [len(str(t).split()) for t in all_texts],
# })
# analysis_df["correct"] = analysis_df["true_label"] == analysis_df["pred_label"]

# correct_len = analysis_df.loc[ analysis_df["correct"], "complaint_length"]
# miscls_len  = analysis_df.loc[~analysis_df["correct"], "complaint_length"]

# print(f"Correctly classified  : {len(correct_len):,}  | mean: {correct_len.mean():.1f} words")
# print(f"Misclassified         : {len(miscls_len):,}  | mean: {miscls_len.mean():.1f} words")

# _, t_p = ttest_ind(correct_len, miscls_len, equal_var=False)
# _, u_p = mannwhitneyu(correct_len, miscls_len, alternative="two-sided")
# n1, n2 = len(correct_len), len(miscls_len)
# pooled_std = np.sqrt(((n1-1)*correct_len.std()**2 + (n2-1)*miscls_len.std()**2) / (n1+n2-2))
# cohens_d   = (miscls_len.mean() - correct_len.mean()) / (pooled_std + 1e-9)

# size = ("Negligible" if abs(cohens_d) < 0.2 else
#         "Small"      if abs(cohens_d) < 0.5 else
#         "Medium"     if abs(cohens_d) < 0.8 else "Large")

# print(f"\nMann-Whitney p = {u_p:.2e}  |  Cohen's d = {cohens_d:.3f} ({size})")
# print(f"Result: Statistically significant (p={u_p:.2e}) but effect size is {size.lower()} "
#       f"(d={cohens_d:.3f}) — complaint length has minimal practical impact on prediction accuracy.")

### 3.10  BERT Model — Civic Agency

To achieve higher classification performance and leverage deep contextual representations, we fine-tune a pre-trained `bert-base-uncased` model using 5-fold cross-validation on Vertex AI.

#### 5-Fold Fine-Tuning Results

The custom Vertex AI training job executed fine-tuning on each fold. The validation metrics achieved on each of the 5 folds are listed below:

| Fold | Validation Accuracy | F1-Macro | Precision | Recall |
| :--- | :---: | :---: | :---: | :---: |
| Fold 0 | 92.05% | 0.8335 | 0.8747 | 0.8034 |
| Fold 1 | 90.83% | 0.8193 | 0.8437 | 0.8034 |
| Fold 2 | 92.23% | 0.8243 | 0.8442 | 0.8144 |
| Fold 3 | 91.40% | 0.8061 | 0.8211 | 0.8012 |
| Fold 4 | 92.04% | 0.8085 | 0.8385 | 0.7909 |
| **Average** | **91.71%** | **0.8184** | **0.8444** | **0.8027** |

*Note: The weighted F1-Macro across folds averages to **0.8184**.*

#### Model Selection: 5-Fold Ensemble Model

Instead of selecting a single model from one of the folds, we will employ an **Ensemble Model** in production.

**Why an Ensemble Model?**
1. **Regularization & Lower Variance:** Deep neural networks can experience variance during fine-tuning on specific folds. Ensembling all 5 fold models by averaging their predicted class probabilities (logits) regularizes predictions and makes the model less sensitive to the noise of any single fold.
2. **Handling Class Imbalance:** Minor categories (such as *KSFES*, *BCP*, or *BESCOM*) have fewer training samples. Certain folds might perform slightly worse on these class representations. An ensemble smooths out these individual weaknesses, resulting in more stable and robust routing across all civic agencies.
3. **Improved Confidence Estimation:** By ensembling the 5 models, the predicted class probabilities are better calibrated, allowing us to implement a confidence threshold for manual routing review of highly uncertain complaints.


### 3.11  Civic Agency Model Comparison


In [ ]:
# import json
# from pathlib import Path

# # Load BERT metrics if available
# bert_metrics = None
# bert_path = Path("metrics_model_civic_bodies/bert/results_bert_civic.json")
# if bert_path.exists():
#     with open(bert_path, "r", encoding="utf-8") as f:
#         bert_res = json.load(f)
#     bert_metrics = {
#         "val_f1_macro": bert_res["avg_val_f1_macro"],
#         "val_accuracy": bert_res["avg_val_accuracy"],
#         "val_precision": bert_res["avg_val_precision"],
#         "val_recall": bert_res["avg_val_recall"]
#     }

# print(f"{'Model':<22} {'F1-Macro':>9} {'Accuracy':>9} {'Precision':>10} {'Recall':>8}")
# print("-"*62)
# for name, m in [("Logistic Regression", lr_metrics),
#                 ("LinearSVC",           svc_metrics),
#                 ("MultinomialNB",       nb_metrics)]:
#     print(f"{name:<22} {m['val_f1_macro']:>9.4f} {m['val_accuracy']:>9.4f} {m['val_precision']:>10.4f} {m['val_recall']:>8.4f}")

# if bert_metrics:
#     print(f"{'BERT (Ensemble)':<22} {bert_metrics['val_f1_macro']:>9.4f} {bert_metrics['val_accuracy']:>9.4f} {bert_metrics['val_precision']:>10.4f} {bert_metrics['val_recall']:>8.4f}")


In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # Per-Agency Accuracy & F1 Heatmaps — LR vs SVC vs NB
# # ─────────────────────────────────────────────────────────────────────────────
# from matplotlib.colors import ListedColormap, BoundaryNorm

# models = {
#     "Logistic Regression": (
#         lambda: Pipeline([
#             ("tfidf", TfidfVectorizer(sublinear_tf=True)),
#             ("clf",   LogisticRegression(solver="lbfgs", class_weight="balanced",
#                                          max_iter=2000, tol=1e-3, n_jobs=-1, random_state=SEED)),
#         ]), lr_params),
#     "LinearSVC": (
#         lambda: Pipeline([
#             ("tfidf", TfidfVectorizer()),
#             ("clf",   LinearSVC(penalty="l2", multi_class="ovr", class_weight="balanced",
#                                 max_iter=2000, dual="auto", random_state=SEED)),
#         ]), svc_params),
#     "MultinomialNB": (
#         lambda: Pipeline([
#             ("tfidf", TfidfVectorizer(sublinear_tf=True)),
#             ("clf",   MultinomialNB()),
#         ]), nb_params),
# }

# # Collect OOF predictions
# agency_reports = {}
# for model_name, (build_fn, params) in models.items():
#     all_true, all_pred = [], []
#     for d in fold_data:
#         m = build_fn()
#         m.set_params(**params)
#         m.fit(d["train_X"], d["train_y"])
#         all_true.extend(d["val_y"])
#         all_pred.extend(m.predict(d["val_X"]))
#     agency_reports[model_name] = classification_report(all_true, all_pred, output_dict=True, zero_division=0)

# # Build DataFrames (values as percentages)
# acc_rows, f1_rows = {}, {}
# for model_name, report in agency_reports.items():
#     acc_rows[model_name] = {ag: report.get(ag, {}).get("precision", 0) * 100 for ag in labels}
#     f1_rows[model_name]  = {ag: report.get(ag, {}).get("f1-score", 0) * 100 for ag in labels}

# df_acc = pd.DataFrame(acc_rows).T
# df_f1  = pd.DataFrame(f1_rows).T

# # Custom colormap: <50 light red, 50-60 light orange, 60-75 orange, 75-90 yellow, 90+ green
# cmap = ListedColormap(["#ffcccc", "#ffd9b3", "#ffb366", "#fff176", "#81c784"])
# bounds = [0, 50, 60, 75, 90, 100]
# norm = BoundaryNorm(bounds, cmap.N)

# fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# # Heatmap 1: Accuracy (Precision)
# sns.heatmap(df_acc, annot=True, fmt=".1f", cmap=cmap, norm=norm,
#             linewidths=1, linecolor="white", cbar=False, ax=axes[0])
# axes[0].set_title("Per-Agency Accuracy (Precision) %", fontsize=14, fontweight="bold")
# axes[0].set_ylabel("")
# axes[0].set_xticklabels(axes[0].get_xticklabels(), fontsize=11)
# axes[0].set_yticklabels(axes[0].get_yticklabels(), fontsize=11, rotation=0)

# # Heatmap 2: F1 Score
# sns.heatmap(df_f1, annot=True, fmt=".1f", cmap=cmap, norm=norm,
#             linewidths=1, linecolor="white", cbar=False, ax=axes[1])
# axes[1].set_title("Per-Agency F1 Score %", fontsize=14, fontweight="bold")
# axes[1].set_ylabel("")
# axes[1].set_xticklabels(axes[1].get_xticklabels(), fontsize=11)
# axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize=11, rotation=0)

# # Legend
# legend_labels = ["< 50%", "50–60%", "60–75%", "75–90%", "≥ 90%"]
# legend_colors = ["#ffcccc", "#ffd9b3", "#ffb366", "#fff176", "#81c784"]
# patches = [plt.Rectangle((0, 0), 1, 1, facecolor=c, edgecolor="gray") for c in legend_colors]
# fig.legend(patches, legend_labels, loc="lower center", ncol=5, fontsize=10,
#            frameon=True, title="Score Range", bbox_to_anchor=(0.5, -0.06))

# plt.tight_layout(h_pad=3)
# plt.savefig(CHARTS_DIR / "3.7_per_agency_accuracy_f1_heatmap.png", dpi=150, bbox_inches="tight")
# plt.show()


---
## 4  Severity Classification — Preprocessing, Augmentation, Training & Inference

**Objective:** Predict complaint urgency — *Low, Medium, High,* or *Critical*.

**Why severity is harder:**
- The linguistic boundary between *High* and *Critical* is subjective.
- *Critical* complaints are rare (class imbalance).
- Misclassifying *Critical* as *Low* has real operational consequences.

**Approach:**  
We optimise for **Critical recall** rather than overall accuracy, using three progressively more targeted strategies:

| Model | Key technique |
|-------|---------------|
| Classical ML (LR, SVC, RF, MNB) | TF-IDF + balanced class weights |
| DistilBERT Model 1 | Weighted Cross-Entropy |
| DistilBERT Model 2 | Focal Loss + ordinal penalty matrix |
| DistilBERT Model 3 | Weighted CE + cascaded waterfall thresholds |
| BiLSTM | Focal Loss + ordinal penalty (native PyTorch) |


### 4.1  Classical Models — Severity Classification


In [ ]:
import joblib
import json
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def _next_version(path: Path, base_name: str) -> int:
    """Return the next unused version index for output files."""
    i = 1
    while (path / f"{base_name}_{i}.json").exists():
        i += 1
    return i

def run_grid_search(model_name, build_fn, param_grid, fold_data, y_global, labels, output_root):
    """
    Cross-validated grid search over param_grid.
    Saves best params, OOF confusion matrix PNG, and metrics JSON.
    Returns (best_params, best_metrics).
    """
    SEED = 43
    out_dir = output_root / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    version = _next_version(out_dir, f"results_{model_name}")
    best_f1, best_params, best_metrics = -1, None, None
    print(f"\n{'='*55}\n  GRID SEARCH — {model_name.upper()}\n{'='*55}")
    for params in ParameterGrid(param_grid):
        random.seed(SEED)
        np.random.seed(SEED)

        # Lists to store metrics per fold
        train_accs, train_precs, train_recs = [], [], []
        train_f1_macros, train_f1_micros, train_f1_weighteds = [], [], []
        
        val_accs, val_precs, val_recs = [], [], []
        val_f1_macros, val_f1_micros, val_f1_weighteds = [], [], []
        for d in fold_data:
            m = build_fn()
            random.seed(SEED)
            np.random.seed(SEED)
            m.set_params(**params)
            m.fit(d["train_X"], d["train_y"])
            
            # ── Train metrics ──
            train_preds = m.predict(d["train_X"])
            train_accs.append(accuracy_score(d["train_y"], train_preds))
            train_precs.append(precision_score(d["train_y"], train_preds, average="macro", zero_division=0))
            train_recs.append(recall_score(d["train_y"], train_preds, average="macro", zero_division=0))
            train_f1_macros.append(f1_score(d["train_y"], train_preds, average="macro"))
            train_f1_micros.append(f1_score(d["train_y"], train_preds, average="micro"))
            train_f1_weighteds.append(f1_score(d["train_y"], train_preds, average="weighted"))
            
            # ── Val metrics ──
            val_preds = m.predict(d["val_X"])
            val_accs.append(accuracy_score(d["val_y"], val_preds))
            val_precs.append(precision_score(d["val_y"], val_preds, average="macro", zero_division=0))
            val_recs.append(recall_score(d["val_y"], val_preds, average="macro", zero_division=0))
            val_f1_macros.append(f1_score(d["val_y"], val_preds, average="macro"))
            val_f1_micros.append(f1_score(d["val_y"], val_preds, average="micro"))
            val_f1_weighteds.append(f1_score(d["val_y"], val_preds, average="weighted"))
        # Averages
        avg_val_f1 = float(np.mean(val_f1_macros))
        print(f"  {params}")
        print(f"    Val  | F1-macro: {avg_val_f1:.4f} | F1-micro: {np.mean(val_f1_micros):.4f} | F1-wgt: {np.mean(val_f1_weighteds):.4f} | Acc: {np.mean(val_accs):.4f} | P: {np.mean(val_precs):.4f} | R: {np.mean(val_recs):.4f}")
        print(f"    Train| F1-macro: {np.mean(train_f1_macros):.4f} | F1-micro: {np.mean(train_f1_micros):.4f} | F1-wgt: {np.mean(train_f1_weighteds):.4f} | Acc: {np.mean(train_accs):.4f} | P: {np.mean(train_precs):.4f} | R: {np.mean(train_recs):.4f}")
        
        if avg_val_f1 > best_f1:
            best_f1 = avg_val_f1
            best_params = params
            best_metrics = {
                "val_f1_macro": float(np.mean(val_f1_macros)),
                "val_f1_micro": float(np.mean(val_f1_micros)),
                "val_f1_weighted": float(np.mean(val_f1_weighteds)),
                "val_accuracy": float(np.mean(val_accs)),
                "val_precision": float(np.mean(val_precs)),
                "val_recall": float(np.mean(val_recs)),
                "train_f1_macro": float(np.mean(train_f1_macros)),
                "train_f1_micro": float(np.mean(train_f1_micros)),
                "train_f1_weighted": float(np.mean(train_f1_weighteds)),
                "train_accuracy": float(np.mean(train_accs)),
                "train_precision": float(np.mean(train_precs)),
                "train_recall": float(np.mean(train_recs)),
            }
    print(f"\n✅ Best Val F1-macro: {best_f1:.4f}  |  Params: {best_params}")
    # OOF confusion matrix
    oof = np.empty_like(y_global, dtype=object)
    for d in fold_data:
        m = build_fn(); m.set_params(**best_params)
        m.fit(d["train_X"], d["train_y"])
        oof[d["val_idx"]] = m.predict(d["val_X"])
    cm = confusion_matrix(y_global, oof, labels=labels)
    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar=False)
    plt.title(f"Average Metrics Confusion Matrix — {model_name} (v{version})")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = out_dir / f"3.7_grid_search_confusion_matrix_{model_name}_{version}.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    results = {
        "version": version, "model": model_name,
        "best_params": {k: list(v) if isinstance(v, tuple) else v for k, v in best_params.items()},
        "metrics": best_metrics,
    }
    with open(out_dir / f"grid_search_results_{model_name}_{version}.json", "w") as fh:
        json.dump(results, fh, indent=4)
    return best_params, best_metrics

SEED = 43
random.seed(SEED)
np.random.seed(SEED)

# Load stratified folds
fold_data_path = PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final.joblib"
cv_folds = joblib.load(fold_data_path)

# Map severity labels to train_y and val_y for run_grid_search compatibility
fold_data_sev = []
for d in cv_folds:
    fold_data_sev.append({
        "train_X": d["train_X"],
        "train_y": d["train_severity"],
        "val_X":   d["val_X"],
        "val_y":   d["val_severity"],
        "val_idx": d["val_idx"]
    })

# Construct y_sev_global using val_idx to align with OOF predictions in run_grid_search
total_size = sum(len(d["val_y"]) for d in fold_data_sev)
y_sev_global = np.empty(total_size, dtype=object)
for d in fold_data_sev:
    y_sev_global[d["val_idx"]] = d["val_y"]

labels_sev   = sorted(np.unique(y_sev_global))
print(f"Severity classes: {labels_sev}")

SEV_OUTPUT = PROJECT_ROOT / "metrics_model_severity"

# ── 5.2.1  Logistic Regression ────────────────────────────────────────────────
lr_sev_params, lr_sev_metrics = run_grid_search(
    model_name="logistic_regression",
    build_fn=lambda: Pipeline([
        ("tfidf", TfidfVectorizer(sublinear_tf=True)),
        ("clf",   LogisticRegression(solver="lbfgs", class_weight="balanced",
                                     max_iter=2000, tol=1e-3, random_state=SEED)),
    ]),
    param_grid={
        "tfidf__max_features": [8_000, 10_000, 12_000],
        "tfidf__ngram_range":  [(1, 1), (1, 2), (1, 3)],
        "clf__C":              [1.0, 2.0, 5.0],
    },
    fold_data=fold_data_sev, y_global=y_sev_global,
    labels=labels_sev, output_root=SEV_OUTPUT,
)

# ── 5.2.2  LinearSVC ──────────────────────────────────────────────────────────
svc_sev_params, svc_sev_metrics = run_grid_search(
    model_name="linearsvc",
    build_fn=lambda: Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf",   LinearSVC(penalty="l2", multi_class="ovr", class_weight="balanced",
                            max_iter=2000, dual="auto", random_state=SEED)),
    ]),
    param_grid={
        "tfidf__max_features": [1_000, 3_000, 5_000],
        "tfidf__ngram_range":  [(1, 1), (1, 2)],
        "clf__C":              [0.1, 1.0, 5.0],
    },
    fold_data=fold_data_sev, y_global=y_sev_global,
    labels=labels_sev, output_root=SEV_OUTPUT,
)

# ── 5.2.3  Multinomial Naive Bayes ────────────────────────────────────────────
mnb_sev_params, mnb_sev_metrics = run_grid_search(
    model_name="multinomialnb",
    build_fn=lambda: Pipeline([
        ("tfidf", TfidfVectorizer(sublinear_tf=True, stop_words="english")),
        ("clf",   MultinomialNB()),
    ]),
    param_grid={
        "tfidf__max_features": [1_000, 3_000, 5_000],
        "tfidf__ngram_range":  [(1, 1), (1, 2)],
        "clf__alpha":          [0.1, 0.5, 1.0, 2.0, 5.0],
    },
    fold_data=fold_data_sev, y_global=y_sev_global,
    labels=labels_sev, output_root=SEV_OUTPUT,
)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  CLASSICAL MODEL COMPARISON — SEVERITY")
print(f"{'='*62}")
print(f"{'Model':<22} {'F1-Macro':>9} {'Accuracy':>9} {'Precision':>10} {'Recall':>8}")
print("-"*62)
for name, m in [("Logistic Regression", lr_sev_metrics),
                ("LinearSVC",           svc_sev_metrics),
                ("Multinomial NB",      mnb_sev_metrics)]:
    print(f"{name:<22} {m['f1_macro']:>9.4f} {m['accuracy']:>9.4f} {m['precision']:>10.4f} {m['recall']:>8.4f}")


### 4.2 BERT Fine-tuning — Severity Classification

Three models are evaluated. Each builds on the previous iteration's failure mode:

**Model 1 — Weighted Cross-Entropy (baseline)**  
Standard fine-tuning with class weights inversely proportional to frequency.

**Model 2 — Focal Loss + Ordinal Penalty**  
Focal Loss down-weights easy examples. The ordinal penalty matrix additionally penalises *under-severity* predictions (predicting Medium for a Critical complaint) more than *over-severity* predictions.

All models use: fp16 mixed precision · gradient checkpointing · fused AdamW · early stopping.


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

# Load stratified folds for BERT (augmented but not preprocessed)
bert_folds_path = PROJECT_ROOT / "data" / "processed" / "cv_fold_data_final_bert.joblib"
cv_folds_bert = joblib.load(bert_folds_path)

# Map severity labels to train_y and val_y for BERT run_grid_search compatibility
fold_data_sev = []
for d in cv_folds_bert:
    fold_data_sev.append({
        "train_X": d["train_X"],
        "train_y": d["train_severity"],
        "val_X":   d["val_X"],
        "val_y":   d["val_severity"],
        "val_idx": d["val_idx"]
    })

# ── Label encoding ────────────────────────────────────────────────────────────
all_labels_bert = []
for fold in fold_data_sev:
    all_labels_bert.extend(fold["train_y"])
    all_labels_bert.extend(fold["val_y"])

le = LabelEncoder()
le.fit(all_labels_bert)
num_labels = len(le.classes_)
print(f"Severity classes ({num_labels}): {list(le.classes_)}")

# Asymmetric class weights: Critical complaints are the most costly to miss
CLASS_WEIGHT_LOOKUP = {"Critical": 4.0, "High": 3.0, "Medium": 2.0, "Low": 1.0}
class_weights_tensor = torch.tensor(
    [CLASS_WEIGHT_LOOKUP.get(lbl, 1.0) for lbl in le.classes_], dtype=torch.float32
)
print("Class weights:", {lbl: CLASS_WEIGHT_LOOKUP.get(lbl, 1.0) for lbl in le.classes_})

tokenizer     = AutoTokenizer.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


class ComplaintDataset(Dataset):
    """Tokenise complaints once; padding is handled dynamically per batch by DataCollator."""
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=256)
        self.labels    = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    prec, rec, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {
        "accuracy":           accuracy_score(labels, preds),
        "f1_macro":           f1_score(labels, preds, average="macro",    zero_division=0),
        "f1_weighted":        f1_w,
        "precision_weighted": prec,
        "recall_weighted":    rec,
    }


_BERT_TRAIN_ARGS = dict(
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    fp16                        = True,
    gradient_checkpointing      = True,
    num_train_epochs            = 3,
    learning_rate               = 1e-4,
    warmup_ratio                = 0.1,
    logging_strategy            = "epoch",
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    greater_is_better           = True,
    max_grad_norm               = 1.0,
    optim                       = "adamw_torch_fused",
    report_to                   = "none",
)


In [ ]:
# ── Weighted Cross-Entropy Trainer ────────────────────────────────────────────
class WeightedCETrainer(Trainer):
    """HuggingFace Trainer with class-weighted cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(model.device)
        )(outputs.logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


# ── Ordinal penalty matrix ────────────────────────────────────────────────────
# Under-severity errors (e.g. predicting Low for Critical) are penalised
# twice as heavily per rank step as over-severity errors.
severity_rank  = {"Critical": 3, "High": 2, "Medium": 1, "Low": 0}
penalty_matrix = torch.zeros((num_labels, num_labels))
for i, true_lbl in enumerate(le.classes_):
    for j, pred_lbl in enumerate(le.classes_):
        diff = severity_rank[true_lbl] - severity_rank[pred_lbl]
        penalty_matrix[i, j] = diff * 2.0 if diff > 0 else abs(diff) * 0.3

print("Ordinal penalty matrix (rows=true, cols=predicted):")
print(pd.DataFrame(penalty_matrix.numpy(), index=le.classes_, columns=le.classes_).round(1))


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss with optional per-class alpha weighting and ordinal penalty.

    gamma  : Focusing parameter (default 2.0). Higher values suppress easy examples.
    alpha  : Per-class weight tensor.
    penalty: (num_labels × num_labels) ordinal penalty matrix.
    """
    def __init__(self, alpha=None, gamma=2.0, penalty=None):
        super().__init__()
        self.alpha   = alpha
        self.gamma   = gamma
        self.penalty = penalty
        self.ce      = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, targets):
        ce_loss = self.ce.to(logits.device)(logits, targets)
        pt      = torch.exp(-ce_loss)
        focal   = (1 - pt) ** self.gamma * ce_loss
        if self.alpha is not None:
            focal = self.alpha.to(logits.device)[targets] * focal
        if self.penalty is not None:
            probs = torch.softmax(logits, dim=1)
            ep    = torch.sum(probs * self.penalty.to(logits.device)[targets], dim=1)
            focal = focal * (1 + ep)
        return focal.mean()


class FocalLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, penalty=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fct = FocalLoss(alpha=class_weights, gamma=gamma, penalty=penalty)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss    = self.loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
# ── Helper: run one DistilBERT cross-validation experiment ────────────────────
def run_bert_cv(trainer_cls, metrics_dir, checkpoint_dir,
                trainer_kwargs=None, compute_metrics_fn=None):
    """
    Runs 5-fold cross-validation for a DistilBERT-based Trainer.
    Returns (fold_results list, cms list).
    """
    metrics_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    if compute_metrics_fn is None:
        compute_metrics_fn = compute_metrics
    if trainer_kwargs is None:
        trainer_kwargs = {}

    all_results, all_cms = [], []

    for fold_id, fold in enumerate(fold_data_sev):
        print(f"\n{'='*40}\n  FOLD {fold_id}\n{'='*40}")

        y_tr = le.transform(fold["train_y"])
        y_va = le.transform(fold["val_y"])

        model = AutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=num_labels
        )
        args  = TrainingArguments(
            output_dir=str(checkpoint_dir / f"fold_{fold_id}"),
            **_BERT_TRAIN_ARGS
        )
        trainer = trainer_cls(
            model=model, args=args,
            train_dataset=ComplaintDataset(fold["train_X"], y_tr),
            eval_dataset =ComplaintDataset(fold["val_X"],   y_va),
            data_collator=data_collator,
            compute_metrics=compute_metrics_fn,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            **trainer_kwargs,
        )
        trainer.train()
        res = trainer.evaluate()
        clean = {k.replace("eval_", ""): round(float(v), 4)
                 for k, v in res.items() if isinstance(v, (int, float))}
        clean["fold"] = fold_id
        all_results.append(clean)

        preds_obj = trainer.predict(ComplaintDataset(fold["val_X"], y_va))
        preds_arr = np.argmax(preds_obj.predictions, axis=1)
        all_cms.append(confusion_matrix(y_va, preds_arr))

        with open(metrics_dir / f"fold_{fold_id}.json", "w") as fh:
            json.dump(clean, fh, indent=4)
        print(f"  Acc {clean['accuracy']:.4f} | F1-Macro {clean['f1_macro']:.4f}")

        del model, trainer
        torch.cuda.empty_cache()

    df_r = pd.DataFrame(all_results)
    summary = {"mean": df_r.mean(numeric_only=True).round(4).to_dict(),
               "std":  df_r.std(numeric_only=True).round(4).to_dict()}
    with open(metrics_dir / "summary.json", "w") as fh:
        json.dump(summary, fh, indent=4)

    avg_cm = np.mean(all_cms, axis=0)
    plt.figure(figsize=(7, 5))
    sns.heatmap(avg_cm, annot=True, fmt=".1f", cmap="Blues", cbar=False,
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f"Average Confusion Matrix — {metrics_dir.name}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(metrics_dir / "average_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\n===== SUMMARY: {metrics_dir.name} =====")
    print(pd.DataFrame(summary))
    return all_results, all_cms


In [ ]:
# ── Model 1: Weighted Cross-Entropy ──────────────────────────────────────────
results_m1, cms_m1 = run_bert_cv(
    trainer_cls   = WeightedCETrainer,
    metrics_dir   = SEV_OUTPUT / "distilbert" / "model_1",
    checkpoint_dir= PROJECT_ROOT / "checkpoints" / "distilbert" / "model_1",
)


#### Observation — Critical Class Recall

The confusion matrix for Model 1 shows that *Critical* complaints are frequently predicted as *High*. This is the most operationally dangerous error.

**Root cause:** Standard weighted cross-entropy still optimises an average loss across all classes. The asymmetric cost of under-severity errors is not explicitly encoded.

**Fix → Model 2:** Focal Loss with an ordinal penalty matrix directly penalises severity underestimation proportional to the rank difference.


In [ ]:
# ── Model 2: Focal Loss + Ordinal Penalty ─────────────────────────────────────
results_m2, cms_m2 = run_bert_cv(
    trainer_cls   = FocalLossTrainer,
    metrics_dir   = SEV_OUTPUT / "distilbert" / "model_2",
    checkpoint_dir= PROJECT_ROOT / "checkpoints" / "distilbert" / "model_2",
    trainer_kwargs= dict(
        class_weights=class_weights_tensor,
        gamma=2.0,
        penalty=penalty_matrix,
    ),
)


#### Model 3 — Cascaded Waterfall Threshold Inference

After evaluating Models 1 and 2, Critical recall still falls short of the 98 % production target. We adopt a hard cascade rule at inference time:

1. If `P(Critical) ≥ t_critical` → Critical  
2. Else if `P(High) ≥ t_high` → High  
3. Else if `P(Medium) ≥ t_medium` → Medium  
4. Otherwise → Low  

`t_critical` is tuned independently per fold using the precision-recall curve targeting ≥ 98 % recall.


In [ ]:
CRITICAL_IDX = list(le.classes_).index("Critical")
HIGH_IDX     = list(le.classes_).index("High")
LOW_IDX      = list(le.classes_).index("Low")
MEDIUM_IDX   = list(le.classes_).index("Medium")

BASE_THRESHOLDS = {"critical": 0.10, "high": 0.15, "medium": 0.20}
TARGET_CRITICAL_RECALL = 0.98


def waterfall_predict(logits: np.ndarray, thresholds: dict = None) -> np.ndarray:
    """Cascade class probabilities in severity order to produce final predictions."""
    if thresholds is None:
        thresholds = BASE_THRESHOLDS
    probs = softmax(logits, axis=1)
    preds = []
    for row in probs:
        if   row[CRITICAL_IDX] >= thresholds["critical"]:
            preds.append(CRITICAL_IDX)
        elif row[HIGH_IDX]     >= thresholds["high"]:
            preds.append(HIGH_IDX)
        elif row[MEDIUM_IDX]   >= thresholds["medium"]:
            preds.append(MEDIUM_IDX)
        else:
            preds.append(LOW_IDX)
    return np.array(preds)


def compute_metrics_waterfall(eval_pred):
    logits, labels = eval_pred
    preds = waterfall_predict(logits)
    prec, rec, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {
        "accuracy":           accuracy_score(labels, preds),
        "f1_macro":           f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted":        f1_w,
        "precision_weighted": prec,
        "recall_weighted":    rec,
    }


In [ ]:
# ── Model 3: Waterfall Threshold ──────────────────────────────────────────────
metrics_dir_3    = SEV_OUTPUT / "distilbert" / "model_3"
checkpoint_dir_3 = PROJECT_ROOT / "checkpoints" / "distilbert" / "model_3"
metrics_dir_3.mkdir(parents=True, exist_ok=True)
checkpoint_dir_3.mkdir(parents=True, exist_ok=True)

all_results_m3, all_cms_m3 = [], []

for fold_id, fold in enumerate(fold_data_sev):
    print(f"\n{'='*40}\n  FOLD {fold_id}  |  Model 3 (Waterfall)\n{'='*40}")

    y_tr = le.transform(fold["train_y"])
    y_va = le.transform(fold["val_y"])

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=num_labels
    )
    args    = TrainingArguments(
        output_dir=str(checkpoint_dir_3 / f"fold_{fold_id}"), **_BERT_TRAIN_ARGS
    )
    trainer = WeightedCETrainer(
        model=model, args=args,
        train_dataset=ComplaintDataset(fold["train_X"], y_tr),
        eval_dataset =ComplaintDataset(fold["val_X"],   y_va),
        data_collator=data_collator,
        compute_metrics=compute_metrics_waterfall,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    # Tune Critical threshold on this fold's validation set
    preds_obj    = trainer.predict(ComplaintDataset(fold["val_X"], y_va))
    raw_logits   = preds_obj.predictions
    val_probs    = softmax(raw_logits, axis=1)
    binary_true  = (y_va == CRITICAL_IDX).astype(int)
    precs_curve, recs_curve, pr_thresholds = precision_recall_curve(
        binary_true, val_probs[:, CRITICAL_IDX]
    )

    # Find the lowest threshold that still achieves the recall target
    optimal_t = BASE_THRESHOLDS["critical"]
    for p, r, t in zip(precs_curve[:-1], recs_curve[:-1], pr_thresholds):
        if r >= TARGET_CRITICAL_RECALL:
            optimal_t = float(t)
            break
    print(f"  Optimal Critical threshold: {optimal_t:.3f}")

    fold_thresholds = {**BASE_THRESHOLDS, "critical": optimal_t}
    preds_arr       = waterfall_predict(raw_logits, fold_thresholds)
    cm              = confusion_matrix(y_va, preds_arr)
    all_cms_m3.append(cm)

    per_class_acc = cm.diagonal() / (cm.sum(axis=1) + 1e-9)
    res = trainer.evaluate()
    clean = {k.replace("eval_", ""): round(float(v), 4)
             for k, v in res.items() if isinstance(v, (int, float))}
    clean["fold"] = fold_id
    clean["critical_threshold"] = round(optimal_t, 4)
    for cls, acc in zip(le.classes_, per_class_acc):
        clean[f"acc_{cls.lower()}"] = round(float(acc), 4)

    all_results_m3.append(clean)
    with open(metrics_dir_3 / f"fold_{fold_id}.json", "w") as fh:
        json.dump(clean, fh, indent=4)
    print(f"  Acc {clean['accuracy']:.4f} | F1-Macro {clean['f1_macro']:.4f} | "
          f"Critical acc {clean.get('acc_critical', 0):.4f}")

    del model, trainer
    torch.cuda.empty_cache()

df_m3 = pd.DataFrame(all_results_m3)
summary_m3 = {"mean": df_m3.mean(numeric_only=True).round(4).to_dict(),
               "std":  df_m3.std(numeric_only=True).round(4).to_dict()}
with open(metrics_dir_3 / "summary.json", "w") as fh:
    json.dump(summary_m3, fh, indent=4)

avg_cm3 = np.mean(all_cms_m3, axis=0)
plt.figure(figsize=(7, 5))
sns.heatmap(avg_cm3, annot=True, fmt=".1f", cmap="Blues", cbar=False,
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Average Confusion Matrix — DistilBERT Model 3 (Waterfall)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(metrics_dir_3 / "average_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n===== MODEL 3 SUMMARY =====")
print(pd.DataFrame(summary_m3))


### 5.4  BiLSTM — Severity Classification

A two-layer bidirectional LSTM using the DistilBERT tokeniser vocabulary. It uses **masked mean pooling** (ignoring padding tokens) and the same Focal Loss + ordinal penalty as Model 2.


In [ ]:
class BiLSTMClassifier(nn.Module):
    """
    Two-layer bidirectional LSTM for text classification.

    Embedding → BiLSTM (×2) → masked mean pooling → Dropout → Linear.
    Masked mean pooling ignores padding positions, producing a cleaner
    sentence representation than using the final hidden state.
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 num_classes=4, num_layers=2, bidirectional=True,
                 dropout=0.3, pad_token_id=0):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                                  bidirectional=bidirectional, batch_first=True, dropout=dropout)
        lstm_out        = hidden_dim * (2 if bidirectional else 1)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_out, num_classes)

    def forward(self, input_ids, attention_mask):
        emb      = self.embedding(input_ids)
        out, _   = self.lstm(emb)
        mask     = attention_mask.unsqueeze(-1).expand(out.size()).float()
        pooled   = torch.sum(out * mask, dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return self.classifier(self.dropout(pooled))


class PaddedDataset(Dataset):
    """Fixed-length padded dataset for native PyTorch DataLoaders."""
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.enc    = tokenizer(list(texts), truncation=True,
                                padding="max_length", max_length=max_length)
        self.labels = labels
    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.enc["input_ids"][idx],      dtype=torch.long),
            "attention_mask": torch.tensor(self.enc["attention_mask"][idx], dtype=torch.long),
            "labels":         torch.tensor(self.labels[idx],                dtype=torch.long),
        }
    def __len__(self):
        return len(self.labels)


In [ ]:
device_lstm = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"BiLSTM device: {device_lstm}")

focal_fn = FocalLoss(
    alpha  = class_weights_tensor.to(device_lstm),
    gamma  = 2.0,
    penalty= penalty_matrix.to(device_lstm),
)

metrics_dir_lstm    = SEV_OUTPUT / "lstm"
checkpoint_dir_lstm = PROJECT_ROOT / "checkpoints" / "lstm"
metrics_dir_lstm.mkdir(parents=True, exist_ok=True)
checkpoint_dir_lstm.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32; MAX_EPOCHS = 100; LR = 1e-4; PATIENCE = 5
all_results_lstm, all_cms_lstm = [], []

for fold_id, fold in enumerate(fold_data_sev):
    print(f"\n{'='*40}\n  FOLD {fold_id}  |  BiLSTM\n{'='*40}")

    y_tr = le.transform(fold["train_y"])
    y_va = le.transform(fold["val_y"])

    train_loader = DataLoader(PaddedDataset(fold["train_X"], y_tr, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(PaddedDataset(fold["val_X"],   y_va, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=False)

    model    = BiLSTMClassifier(vocab_size=tokenizer.vocab_size, num_classes=num_labels,
                                 pad_token_id=tokenizer.pad_token_id).to(device_lstm)
    optim    = torch.optim.AdamW(model.parameters(), lr=LR)
    scheduler= torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.1, patience=2)

    best_f1, best_state, no_improve = 0.0, copy.deepcopy(model.state_dict()), 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for batch in train_loader:
            ids    = batch["input_ids"].to(device_lstm)
            mask   = batch["attention_mask"].to(device_lstm)
            labels = batch["labels"].to(device_lstm)
            optim.zero_grad()
            loss = focal_fn(model(ids, mask), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        model.eval()
        val_preds, val_targets, val_loss_sum = [], [], 0.0
        with torch.no_grad():
            for batch in val_loader:
                ids    = batch["input_ids"].to(device_lstm)
                mask   = batch["attention_mask"].to(device_lstm)
                labels = batch["labels"].to(device_lstm)
                logits = model(ids, mask)
                val_loss_sum += focal_fn(logits, labels).item()
                val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                val_targets.extend(batch["labels"].numpy())

        avg_loss = val_loss_sum / len(val_loader)
        val_f1   = f1_score(val_targets, val_preds, average="macro", zero_division=0)
        val_acc  = accuracy_score(val_targets, val_preds)
        print(f"  Epoch {epoch+1:3d} | Loss {avg_loss:.4f} | F1 {val_f1:.4f} | Acc {val_acc:.4f}")

        scheduler.step(avg_loss)
        if val_f1 > best_f1:
            best_f1, best_state, no_improve = val_f1, copy.deepcopy(model.state_dict()), 0
            torch.save(best_state, checkpoint_dir_lstm / f"best_fold_{fold_id}.pt")
            print("    >>> Best saved.")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print("    >>> Early stopping."); break

    model.load_state_dict(best_state)
    model.eval()
    final_preds, final_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(device_lstm)
            mask = batch["attention_mask"].to(device_lstm)
            final_preds.extend(torch.argmax(model(ids, mask), dim=1).cpu().numpy())
            final_targets.extend(batch["labels"].numpy())

    prec, rec, f1_w, _ = precision_recall_fscore_support(
        final_targets, final_preds, average="weighted", zero_division=0
    )
    clean = {
        "accuracy":    round(accuracy_score(final_targets, final_preds), 4),
        "f1_macro":    round(f1_score(final_targets, final_preds, average="macro", zero_division=0), 4),
        "f1_weighted": round(f1_w, 4), "precision_weighted": round(prec, 4),
        "recall_weighted": round(rec, 4), "fold": fold_id,
    }
    all_results_lstm.append(clean)
    all_cms_lstm.append(confusion_matrix(final_targets, final_preds))
    with open(metrics_dir_lstm / f"fold_{fold_id}.json", "w") as fh:
        json.dump(clean, fh, indent=4)
    print(f"\n  [Fold {fold_id}] Acc {clean['accuracy']:.4f} | F1-Macro {clean['f1_macro']:.4f}")

df_lstm = pd.DataFrame(all_results_lstm)
summary_lstm = {"mean": df_lstm.mean(numeric_only=True).round(4).to_dict(),
                "std":  df_lstm.std(numeric_only=True).round(4).to_dict()}
with open(metrics_dir_lstm / "summary.json", "w") as fh:
    json.dump(summary_lstm, fh, indent=4)

avg_cm_lstm = np.mean(all_cms_lstm, axis=0)
plt.figure(figsize=(7, 5))
sns.heatmap(avg_cm_lstm, annot=True, fmt=".1f", cmap="Blues", cbar=False,
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Average Confusion Matrix — BiLSTM")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(metrics_dir_lstm / "average_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n===== BiLSTM SUMMARY =====")
print(pd.DataFrame(summary_lstm))


### 5.5  Save Best DistilBERT Severity Model


In [ ]:
best_fold_data = max(all_results_m3, key=lambda x: x["f1_macro"])
best_fold_id   = best_fold_data["fold"]
print(f"Best fold: {best_fold_id}  |  F1-Macro: {best_fold_data['f1_macro']:.4f}")

best_fold_ckpt_dir = checkpoint_dir_3 / f"fold_{best_fold_id}"
checkpoints = sorted(
    [f for f in best_fold_ckpt_dir.iterdir()
     if f.is_dir() and f.name.startswith("checkpoint-")],
    key=lambda x: int(x.name.split("-")[1]),
)

with open(checkpoints[-1] / "trainer_state.json") as fh:
    trainer_state = json.load(fh)
best_ckpt_path = trainer_state["best_model_checkpoint"]
print(f"Loading checkpoint: {best_ckpt_path}")

PROD_DIR = SEV_OUTPUT / "distilbert" / "production_model"
PROD_DIR.mkdir(parents=True, exist_ok=True)

prod_model = AutoModelForSequenceClassification.from_pretrained(best_ckpt_path)
prod_model.save_pretrained(PROD_DIR)
tokenizer.save_pretrained(PROD_DIR)
joblib.dump(le, PROD_DIR / "label_encoder.joblib")

print(f"\n✅ Production model saved to: {PROD_DIR}")
print("   Artefacts: model weights · tokenizer config · label encoder")


### 5.6  Inference — Predict Severity of New Complaints


In [ ]:
infer_tokenizer = AutoTokenizer.from_pretrained(PROD_DIR)
infer_model     = AutoModelForSequenceClassification.from_pretrained(PROD_DIR)
infer_le        = joblib.load(PROD_DIR / "label_encoder.joblib")

infer_device = "cuda" if torch.cuda.is_available() else "cpu"
infer_model.to(infer_device).eval()
print(f"✅ Inference model ready on {infer_device}.")


def predict_severity(text: str) -> dict:
    """
    Predict the severity of a complaint string.

    Returns the predicted label and the full probability distribution
    over all severity classes.
    """
    inputs = infer_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=256
    ).to(infer_device)
    with torch.no_grad():
        logits = infer_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id    = int(np.argmax(probs))
    pred_label = infer_le.inverse_transform([pred_id])[0]
    return {
        "predicted_severity": pred_label,
        "probabilities": {cls: round(float(p), 4)
                          for cls, p in zip(infer_le.classes_, probs)},
    }


samples = [
    "The entire water supply to our area has been cut for 3 days. Families are suffering.",
    "There are potholes on the road near my house.",
    "Street light outside my building is not working for the past week.",
    "Garbage has not been collected from our locality for the past month. There is a foul smell.",
]

print("\n" + "="*60)
print("  SEVERITY PREDICTIONS")
print("="*60)
for text in samples:
    r = predict_severity(text)
    print(f"\n📋 Complaint : {text}")
    print(f"   Severity  : {r['predicted_severity']}")
    print(f"   Probs     : {r['probabilities']}")


---
## Summary & Next Steps

### Model Performance at a Glance

| Task | Best Model | Key Metric |
|------|-----------|------------|
| Civic Agency Routing | LinearSVC (TF-IDF) | F1-Macro — see `metrics_model_civic_bodies/` |
| Severity Classification | DistilBERT Model 3 (Waterfall) | Critical Recall ≥ 98 % — see `metrics_model_severity/distilbert/model_3/` |

### Key Design Decisions

- **Stratified fold-level augmentation** — augmented only inside each training fold, preventing any test-set contamination from synthetic data.
- **Cascaded waterfall inference** with adaptive per-fold Critical threshold — ensures Critical complaints are almost never silently downgraded at the cost of slightly more false Critical positives.
- **Ordinal penalty matrix** — encodes domain knowledge that under-severity errors (Critical predicted as Low) are far costlier than over-severity errors, directly shaping the loss landscape.

### Recommended Next Steps

1. **Threshold recalibration** — revisit the Critical threshold once production complaint volumes are available; a lower False Positive Rate may be acceptable if operator workload allows it.
2. **Active learning loop** — route low-confidence predictions to human reviewers; use corrections to incrementally fine-tune the model.
3. **Multi-label extension** — some complaints span multiple agencies or severity levels; a multi-label architecture would handle these more gracefully.
4. **Monitoring & drift detection** — track prediction probability distributions over time to catch shifts in complaint language (e.g. new issues emerging after a city-wide event).
